# Ноутбук 3. Разложение на множители и мультипликативная группа $(\mathbb Z/n\mathbb Z)^*$

В ноутбуке 1 мы научились быстро вычислять $a^{-1}\bmod n$ и $a^e\bmod n$, в ноутбуке 2 — работать с простыми числами. Теперь соберём из этого ту алгебраическую конструкцию, в которой живут протоколы Диффи—Хеллмана, Эль-Гамаля и RSA: **группу обратимых классов вычетов** $(\mathbb Z/n\mathbb Z)^*$.

Нас интересуют четыре вопроса:

1. как разложить число на простые множители и сколько это стоит;
2. сколько элементов в группе $(\mathbb Z/n\mathbb Z)^*$ (функция Эйлера $\varphi$) и почему $\varphi(n)$ — секрет RSA;
3. что такое порядок элемента и как его найти быстро;
4. когда вся группа порождается одним элементом (первообразный корень) и почему на этом построен дискретный логарифм.

По каждой теме идём одним и тем же путём:

$$
\text{идея}\;\to\;\text{как вычисляем}\;\to\;\text{сколько это стоит}\;\to\;\text{где это в криптографии}.
$$

Стоимость, как и раньше, измеряем относительно битовой длины входа $L=\lfloor\log_2 n\rfloor+1$. Главный урок ноутбука — увидеть на своём компьютере границу между тем, что вычисляется за полиномиальное по $L$ время (степень, обратный элемент, порядок при известном разложении), и тем, что требует экспоненциального перебора (разложение $n=pq$, дискретный логарифм).

Сравнения, классы вычетов, полная система вычетов, обратный элемент и китайская теорема об остатках уже разобраны в ноутбуке 1 и здесь не повторяются: в тексте даются ссылки на соответствующие определения и формулы.

<div style="background-color: #1a191c; padding: 20px; border-radius: 10px; border-left: 4px solid #7653a6;">

### 🧩 Что появится к концу ноутбука

```text
Разложение и мультипликативные функции     Группа (Z/nZ)*
├── smallest_factor()                       class ZnZ
├── decompose(), recompose()                ├── elems(), group_order()
├── mult_function()                         ├── multiplicative_order()
├── tau, sigma                              ├── subgroup(), is_cyclic()
└── euler_phi()                             └── primitive_roots()

Быстрые алгоритмы и криптография
├── multiplicative_order_fast()
├── is_primitive_root(), primitive_root()
└── discrete_log_bruteforce()
```

В начале ноутбука (задание 3.0) в `crypto_math` переносятся `is_prime_fast`, `primes` и `next_prime` из ноутбука 2. Несколько функций этого ноутбука в конце тоже переносятся в `crypto_math`, потому что понадобятся в ноутбуках о Диффи—Хеллмане и RSA. Класс `ZnZ` остаётся учебным инструментом этого ноутбука.

</div>

### Терминология

- **разложение на простые множители** (*prime factorization*);
- **мультипликативная функция** (*multiplicative function*);
- **функция Эйлера** (*Euler's totient function*);
- **приведённая система вычетов** (*reduced residue system*);
- **порядок группы** (*order of a group*) и **порядок элемента** (*multiplicative order*);
- **циклическая группа**, **образующий элемент** (*cyclic group, generator*);
- **первообразный корень** (*primitive root*);
- **дискретный логарифм** (*discrete logarithm*);
- **безопасное простое число** (*safe prime*).

<div style="background-color: #1f2122; padding: 20px; border-radius: 10px; border-left: 4px solid #007cba;">

### Задание 3.0. Что берём из ноутбуков 1 и 2

Чтобы не переписывать в этом ноутбуке уже готовый код, всё нужное из прошлых ноутбуков импортируем из `crypto_math`.

**Уже в библиотеке** (перенесено в ноутбуке 2 из ноутбука 1): `gcd`, `mod_inverse`, `mod_pow`, `crt`, `random_exact_bits`. Здесь они нужны для группы $(\mathbb Z/n\mathbb Z)^*$, функции Эйлера, порядка элемента и метода Полига—Хеллмана (задание 3.19).

**Переносим сейчас из ноутбука 2:**

| функция | где написана в ноутбуке 2 | где нужна в этом ноутбуке |
|---|---|---|
| `is_prime_fast(n)` | задание 3 | случайные простые `random_prime` (раздел 3.1), безопасное простое (задание 3.20) |
| `primes(n)` | задание 4 | список простых $p<10^5$ (исследование 3.6) |
| `next_prime(n)` | задание 17 | поиск безопасного простого (задание 3.20) |

Остальное из ноутбука 2 здесь не требуется: `is_prime` был учебным первым вариантом, а `isprime_list` и `prime_range` прямо не вызываются. Решето из ноутбука 2 в задании 3.2 не импортируется, а переделывается в новую функцию `omega_list`.

1. Скопируйте `is_prime_fast`, `primes` и `next_prime` в `crypto_math/number_theory.py` так же, как в ноутбуке 2 переносились функции ноутбука 1 (шаги 3–4 того ноутбука). Отладочные `print` в библиотеку не переносите.
2. Добавьте в `tests/test_number_theory.py` проверки свойств, а не отдельных значений: `is_prime_fast` и `primes` согласованы на $0\le n\le10^4$; `next_prime(n)` простое, больше $n$, и между ними простых нет.
3. Запустите `pytest -q`, перезапустите kernel и выполните ячейку ниже.

</div>

In [1]:
import math
import random
import time
from collections import Counter
from math import isqrt

from crypto_math.number_theory import (
    gcd, mod_inverse, mod_pow, crt, random_exact_bits,   # из ноутбука 1
    is_prime_fast, primes, next_prime,                   # из ноутбука 2
)

assert gcd(30, 18) == 6
assert mod_inverse(7, 23) == 10
assert mod_pow(7, 13, 23) == pow(7, 13, 23)
assert [n for n in range(50) if is_prime_fast(n)] == primes(49)
assert len(primes(10**5)) == 9592
assert next_prime(1) == 2 and next_prime(2) == 3 and next_prime(13) == 17
print("crypto_math подключена")

crypto_math подключена


## 3.1. Разложение на простые множители

*Материал разделов 3.1–3.2 (функции `smallest_factor`, `decompose`, `mult_function` и часть заданий) переработан из ноутбука P3wNT Notebook 4 М. Х. Вайсмана, сопровождающего книгу* An Illustrated Theory of Numbers *[7].*

### Основная теорема арифметики

Всякое натуральное $n>1$ раскладывается в произведение простых чисел, и это разложение единственно с точностью до порядка множителей:

$$
n=p_1^{e_1}p_2^{e_2}\cdots p_k^{e_k},\qquad p_1<p_2<\cdots<p_k,\quad e_i\ge1.
\tag{3.1}
$$

Запись (3.1) называется **каноническим разложением** числа $n$.

В Python каноническое разложение удобно хранить словарём: простые множители — ключи, показатели — значения. Например, $2^3\cdot3^2\cdot7$ хранится как `{2: 3, 3: 2, 7: 1}`, а числу $1$ соответствует пустой словарь `{}`.

### Как раскладываем: пробное деление

Стратегия простая: находим наименьший простой делитель $p$ числа, делим на него, записываем $p$ в словарь и повторяем, пока число не станет равным $1$.

Наименьший делитель ищем перебором. Тот же довод, что и в задании 3 ноутбука 2, позволяет остановиться на $\sqrt n$: если у $n$ нет делителя $d\le\sqrt n$, то $n$ простое. Граница вычисляется через `math.isqrt`, а после двойки перебираются только нечётные кандидаты.

In [2]:
# Пробное деление. У составного n = a*b, где 1 < a <= b, обязательно
# есть делитель a <= floor(sqrt(n)). Если такого нет и n > 1, число простое.
# Наименьший делитель больше 1 всегда прост: иначе у него был бы делитель ещё меньше.
def smallest_factor(n):
    '''Наименьший простой делитель n >= 2; для простого n возвращает само n.'''
    # У 0, 1 и отрицательных простого делителя нет.
    if n < 2:
        return None
    # Единственное чётное простое — 2. Любое чётное n > 2 делится на 2.
    if n % 2 == 0:
        return 2
    # isqrt — точный floor(sqrt(n)). Шаг 2: после двойки чётные кандидаты не нужны,
    # чётное d > 2 само делится на 2 и не может быть наименьшим делителем нечётного n.
    for d in range(3, isqrt(n) + 1, 2):
        if n % d == 0:
            return d
    # Делителя до sqrt(n) нет, значит n само простое и является своим наименьшим делителем.
    return n


# Каноническое разложение (3.1): n = p1^e1 * ... * pk^ek, простые по возрастанию.
# В словаре ключ — простое, значение — показатель. У 1 множителей нет, словарь пустой.
def decompose(N):
    '''Каноническое разложение N >= 1 в виде словаря {p: e}.'''
    n = N
    D = {}
    while n > 1:
        p = smallest_factor(n)
        # Один и тот же p может делить n несколько раз: у 12 двойка встречается дважды.
        # get(p, 0) начинает показатель с нуля, если p ещё не встречалось.
        D[p] = D.get(p, 0) + 1
        n //= p
    return D


print(smallest_factor(105), smallest_factor(1999**2), smallest_factor(101))
print(decompose(100))
print(decompose(56401910421778813463))   # 11^3 * 13^9 * 1999^2
print(decompose(1))

3 1999 101
{2: 2, 5: 2}
{11: 3, 13: 9, 1999: 2}
{}


Наименьший делитель всегда прост: если бы он раскладывался как $d=ab$ с $1<a<d$, то $a$ был бы меньшим делителем $n$. Поэтому `decompose` действительно возвращает простые множители.

Заметим, что 66-битное число $56401910421778813463$ разложилось мгновенно: у него маленькие простые множители, и перебор каждый раз останавливается рано. Насколько всё меняется, когда маленьких множителей нет, покажет исследование 3.1.

<div style="background-color: #17191b; padding: 20px; border-radius: 10px; border-left: 4px solid #007cba;">

### Задание 3.1. Обратная операция

Напишите функцию `recompose(D)`, которая восстанавливает число по словарю разложения. Проверьте, что `recompose(decompose(n)) == n` для всех $1\le n\le 10\,000$.

```python
D = decompose(1000)
print(D)
print(recompose(D))
print(recompose({2: 1, 3: 1, 5: 1, 7: 1}))
```

Ожидаемый вывод:

```text
{2: 3, 5: 3}
1000
210
```

</div>

In [3]:
# Ваше решение:

# Обратная операция к decompose. Словарь {p: e} собирается обратно в число
# произведением p^e. Пустой словарь — это 1: произведение пустого набора равно 1,
# и именно с единицы мы начинаем.

def recompose(D):
    """Число по словарю канонического разложения. Пустой словарь — это 1."""
    # 1 — нейтральный элемент умножения, поэтому степень каждого p дописывается к ней.
    number = 1
    for prime, exponent in D.items():
        number *= prime ** exponent
    return number


# 1000 = 10^3 = (2*5)^3 = 2^3 * 5^3. Второе число — произведение первых четырёх простых, 210.
D = decompose(1000)
print(D)
print(recompose(D))
print(recompose({2: 1, 3: 1, 5: 1, 7: 1}))

# Круговая проверка на всём отрезке: разложить и собрать должно вернуть то же n.
# Если бы показатель потерялся или простое перепуталось, равенство сломалось бы.
mismatches = [n for n in range(1, 10_001) if recompose(decompose(n)) != n]
print(f"recompose(decompose(n)) == n на 1..10000, расхождений: {len(mismatches)}")
assert not mismatches


{2: 3, 5: 3}
1000
210
recompose(decompose(n)) == n на 1..10000, расхождений: 0


<div style="background-color: #242729; padding: 20px; border-radius: 10px; border-left: 4px solid #007cba;">

### Задание 3.2. Сколько различных простых делителей у типичного числа

Обозначим через $\omega(n)$ число различных простых делителей $n$, то есть `len(decompose(n))`; $\omega(1)=0$.

**Гипотеза.** До вычислений предскажите: какое значение $\omega(n)$ встречается чаще всего среди $1\le n\le 10^6$? Бывают ли там числа с $\omega(n)=8$?

Найдите распределение чисел $1\le n\le10^6$ по значению $\omega(n)$ и заполните таблицу.

| $\omega(n)$ | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| количество $n$ | | | | | | | | |

Для самопроверки: $\omega(n)=0$ только у $n=1$, а $\omega(n)=7$ ровно у 8 чисел этого диапазона.

Напишите функцию `omega_list(n)` по алгоритму, представленному на блок-схеме.

**Вопросы.**

1. Почему $\omega(n)=8$ в этом диапазоне невозможно? Укажите наименьшее число с восемью различными простыми делителями.
2. Чем числа с $\omega(n)=1$ отличаются от простых чисел? Сравните их количество с $\pi(10^6)=78\,498$ из ноутбука 2.

*Подсказка.* Вызов `decompose` для каждого $n$ до $10^6$ займёт от нескольких секунд до минуты в зависимости от компьютера. Быстрее изменить решето Эратосфена из ноутбука 2: для каждого простого $p$ прибавлять единицу ко всем кратным $p$. Если к моменту прихода в $p$ значение `om[p]` всё ещё $0$, то $p$ простое: составное число уже получило единицу от своего меньшего простого делителя. Каждое $n$ получает по единице от каждого своего простого делителя, поэтому в конце `om[n]` $=\omega(n)$, а таблицу даёт `Counter(om[1:])`. Например, для $N=12$ список `om[1:]` равен `[0, 1, 1, 1, 1, 2, 1, 1, 1, 2, 1, 2]`. Отличия от `isprime_list` из ноутбука 2: вместо вычёркивания прибавляем единицу, $p$ идёт до $N$, а не до $\sqrt N$, и кратные берутся начиная с самого $p$, а не с $p^2$.
<img src="attachment:omega-flowchart.png" alt="Блок-схема функции omega_list(n)" width="500">
</div>

In [4]:
# Ваше решение:

# ω(n) — число различных простых делителей n. ω(1) = 0: у единицы их нет.
# Обычный decompose для каждого n до 10^6 слишком дорог. Здесь то же решето,
# что в ноутбуке 2, но вместо вычёркивания каждому кратному добавляется 1.

def omega_list(n):
    """om[k] = ω(k) для 0 <= k <= n. Ноль у индекса 0 не используется, ω(1) = 0.

    Тот же проход, что у решета: если к моменту p значение om[p] ещё 0, p простое.
    Каждому кратному, начиная с самого p, добавляется единица. p идёт до n, а не до √n.
    """
    # om[k] копится в ω(k). Индекс 0 не используется: числа начинаются с 1.
    # Изначально все нули, в том числе om[1]: единицу ни одно простое не кратно.
    om = [0] * (n + 1)
    # p идёт до n, а не до sqrt(n). Иначе большие простые не успели бы
    # прибавить единицу своим кратным, и ω получилась бы заниженной.
    for p in range(2, n + 1):
        # Если к моменту прихода в p счётчик всё ещё 0, меньший простой p не делил.
        # Значит p само простое. Составное уже получило единицу от своего меньшего делителя.
        if om[p] == 0:
            # Кратные начинаются с самого p, не с p^2: самому простому тоже нужна единица.
            # Каждое n получает ровно по одной единице от каждого своего простого делителя.
            for multiple in range(p, n + 1, p):
                om[multiple] += 1
    return om


# Подсказка ноутбука: для N = 12 список om[1:] равен
# [0, 1, 1, 1, 1, 2, 1, 1, 1, 2, 1, 2]. Это ω(1), ..., ω(12).
assert omega_list(12)[1:] == [0, 1, 1, 1, 1, 2, 1, 1, 1, 2, 1, 2]

print(
    "Гипотеза до счёта. Средний порядок ω(n) около log log n.\n"
    "log log(10^6) ≈ 2.6, поэтому чаще всего жду 2 или 3.\n"
    "Восемь различных простых множителей в 10^6, скорее всего, не помещаются:\n"
    "произведение первых восьми простых уже больше миллиона."
)

# Считаем распределение до печати таблицы, но гипотезу формулируем заранее, в print выше.
# Counter(om[1:]) группирует числа по значению ω. Ноль индекса 0 в подсчёт не входит.
N = 10**6
om = omega_list(N)
counts = Counter(om[1:])

print("\nω(n) | количество n из 1..10^6")
for omega in range(0, max(counts) + 1):
    print(f"{omega:5d} | {counts[omega]}")

# Самопроверка из задания: ω = 0 только у n = 1, ω = 7 ровно у 8 чисел, ω = 8 не бывает.
assert counts[0] == 1
assert counts[7] == 8
assert counts.get(8, 0) == 0

# Самое частое значение ω — ключ с наибольшим счётчиком.
mode = max(counts, key=counts.get)
print(f"\nЧаще всего встречается ω(n) = {mode}: таких чисел {counts[mode]}.")

# Числа с ω(n) = 1 — это степени простых p^e, e >= 1.
# Среди них простые (e = 1) и высшие степени p^2, p^3, ... (e >= 2).
# Высшую степень перечисляем явно: p^2, p^3, ... пока не выйдем за N.
# p^e при e >= 2 не больше N только если p <= sqrt(N).
higher_powers = 0
for prime in range(2, isqrt(N) + 1):
    if not is_prime_fast(prime):
        continue
    power = prime * prime
    while power <= N:
        higher_powers += 1
        power *= prime

# π(10^6) = 78498 дано в ноутбуке 2. Проверка: число степеней простых
# равно числу простых плюс число высших степеней.
pi_million = 78_498
assert counts[1] == pi_million + higher_powers

# Наименьшее число с восемью различными простыми делителями — произведение
# первых восьми простых. Любое другое такое число только больше.
primorial_8 = 2 * 3 * 5 * 7 * 11 * 13 * 17 * 19
print(
    f"\n1. ω(n) = 8 на 1..10^6 невозможно. Наименьшее число с восемью различными\n"
    f"   простыми делителями — произведение первых восьми простых, {primorial_8}.\n"
    f"   Любое другое такое число только больше, а {primorial_8} > 10^6.\n"
    f"\n2. ω(n) = 1 означает степень простого p^e, e >= 1: само простое и p^2, p^3, ...\n"
    f"   Таких чисел {counts[1]}. Простых среди них π(10^6) = {pi_million},\n"
    f"   остальные {higher_powers} — это p^e с показателем e >= 2.\n"
    f"   Гипотеза про моду совпала: чаще всего ω = {mode}."
)


Гипотеза до счёта. Средний порядок ω(n) около log log n.
log log(10^6) ≈ 2.6, поэтому чаще всего жду 2 или 3.
Восемь различных простых множителей в 10^6, скорее всего, не помещаются:
произведение первых восьми простых уже больше миллиона.

ω(n) | количество n из 1..10^6
    0 | 1
    1 | 78734
    2 | 288726
    3 | 379720
    4 | 208034
    5 | 42492
    6 | 2285
    7 | 8

Чаще всего встречается ω(n) = 3: таких чисел 379720.

1. ω(n) = 8 на 1..10^6 невозможно. Наименьшее число с восемью различными
   простыми делителями — произведение первых восьми простых, 9699690.
   Любое другое такое число только больше, а 9699690 > 10^6.

2. ω(n) = 1 означает степень простого p^e, e >= 1: само простое и p^2, p^3, ...
   Таких чисел 78734. Простых среди них π(10^6) = 78498,
   остальные 236 — это p^e с показателем e >= 2.
   Гипотеза про моду совпала: чаще всего ω = 3.


Для следующих экспериментов нужны случайные простые числа заданной битовой длины. Простоту проверяем функцией `is_prime_fast` из ноутбука 2 (задание 3.0): это то же пробное деление до $\sqrt n$, что и `smallest_factor(n) == n`. Для чисел до 30–32 бит этого достаточно; быстрые тесты для больших простых появятся в ноутбуке 4.

In [5]:
# Случайное простое ровно из bits бит. random_exact_bits уже ставит старший бит,
# поэтому длина равна bits. «| 1» делает число нечётным: чётное больше 2 не простое.
# Для 30–32 бит пробного деления is_prime_fast хватает. Отклоняем кандидата,
# у которого | 1 случайно увеличил бы длину (на практике старший бит уже стоит).
def random_prime(bits):
    '''Случайное простое число ровно из bits бит (bits >= 2, небольшие bits).'''
    while True:
        p = random_exact_bits(bits) | 1
        if p.bit_length() == bits and is_prime_fast(p):
            return p


p = random_prime(24)
print(p, p.bit_length(), is_prime_fast(p))

13859579 24 True


<div style="background-color: #181817; padding: 20px; border-radius: 10px; border-left: 4px solid #d39e00;">

### 🔬 Исследование 3.1. Сколько стоит разложить $n=pq$

В RSA модуль $n=pq$ — произведение двух простых **одинаковой длины**. Посмотрим, как растёт время работы `decompose` на таких числах.

**Гипотеза.** Если $p\approx q\approx\sqrt n$, то `smallest_factor` перебирает около $\sqrt n/2$ нечётных кандидатов. Во сколько раз должно вырасти время, если $L$ увеличить на $4$ бита?

**Ход.** Для каждого $L$ из таблицы трижды сгенерируйте $p$ и $q$ по $L/2$ бит (`random_prime(L // 2)`), проверьте, что `(p * q).bit_length()` равно $L$ или $L-1$, замерьте время `decompose(p * q)` через `time.perf_counter()` и запишите медиану.

**Часть A. Замеры на вашем ПК.**

| $L$, бит | медиана времени, с | отношение к предыдущей строке |
|---:|---:|---:|
| 32 | | — |
| 36 | | |
| 40 | | |
| 44 | | |
| 48 | | |
| 52 | | |

**Часть B. Экстраполяция.** Время пробного деления растёт примерно как $t_0\cdot2^{L/2}$, где $t_0$ — время (в секундах) на одного кандидата-делителя. Найдите $t_0$ по последней строке таблицы и оцените время разложения 512-, 1024- и 2048-битного модуля этим методом. Переведите ответ в годы и сравните с возрастом Вселенной (около $4\cdot10^{17}$ с).

**Вопросы.**

1. Совпало ли наблюдаемое отношение с вашей гипотезой?
2. Почему число $56401910421778813463$ из примера выше разложилось мгновенно, хотя оно длиннее всех чисел таблицы? Какие простые $p,q$ поэтому нельзя брать в RSA?
3. Пробное деление — самый простой, но не лучший метод (обзор методов — [3, §2.3, с. 228–254]; пробное деление — [3, §2.3.2, с. 232], [5, §14.1, с. 319]; число пробных делений на примере — [6, табл. 3.2, с. 50–51]). Лучшие известные алгоритмы разложения работают быстрее любой экспоненты $2^{cL}$, но всё же не полиномиально по $L$. Почему криптографу важно именно это различие, а не скорость конкретной реализации?

</div>

In [6]:
# Выполните исследование здесь:

from statistics import median
import math

print(
    "Гипотеза. У модуля n = pq с простыми по L/2 бит наименьший делитель около √n.\n"
    "Перебор в smallest_factor доходит примерно до 2^{L/2}.\n"
    "Плюс 4 бита умножают эту границу на 4, значит и время должно вырасти примерно в 4 раза."
)


# Один замер — это случайный модуль RSA длины около L: два простых по L/2 бит.
# Медиана трёх запусков, а не среднее: один неудачный простой не должен
# тянуть оценку, если у него случайно оказался маленький делитель.
def median_factor_time(L, trials=3):
    samples = []
    for _ in range(trials):
        while True:
            p = random_prime(L // 2)
            q = random_prime(L // 2)
            modulus = p * q
            # Произведение двух L/2-битных чисел имеет длину L или L-1:
            # переполнение старшего бита зависит от того, насколько простые близки к 2^{L/2}.
            if modulus.bit_length() in (L, L - 1):
                break
        # В замер входит только decompose. Генерация простых — отдельная работа,
        # её время к стоимости разложения не относится.
        started = time.perf_counter()
        factors = decompose(modulus)
        samples.append(time.perf_counter() - started)
        assert recompose(factors) == modulus
    return median(samples)


lengths = [32, 36, 40, 44, 48, 52]
medians = []
print(f"\n{'L':>4} {'медиана, с':>14} {'отношение':>12}")
previous = None
for L in lengths:
    duration = median_factor_time(L)
    medians.append(duration)
    ratio = "—" if previous is None else f"{duration / previous:.2f}"
    print(f"{L:4d} {duration:14.4f} {ratio:>12}")
    previous = duration

ratios = [medians[i] / medians[i - 1] for i in range(1, len(medians))]
last_L = lengths[-1]
last_time = medians[-1]
# Формула задания: t ≈ t0 * 2^{L/2}. t0 — цена одного кандидата вместе с множителем 1/2
# от того, что после двойки перебираются только нечётные.
# Формула задания: время пробного деления t ≈ t0 * 2^{L/2}.
# 2^{L/2} — это порядок sqrt(n), то есть сколько кандидатов смотрит smallest_factor,
# когда меньший множитель сам около sqrt(n). t0 вытаскиваем из последней строки:
# там замер самый длинный и меньше всего похож на шум таймера.
# Для 512, 1024 и 2048 бит 2^{L/2} не помещается в обычное представление «секунды · число»,
# поэтому печатаем десятичный логарифм: 10^k секунд, лет и возрастов Вселенной.
t0 = last_time / 2 ** (last_L / 2)
print(f"\nt0 по строке L = {last_L}: {t0:.3e} с.")
print(f"{'L':>6} {'секунды':>12} {'годы':>12} {'возрасты Вселенной':>22}")
# 365.25 суток — средний год с високосными. 4·10^17 с — возраст Вселенной из задания.
log10_year = math.log10(365.25 * 24 * 3600)
log10_universe = math.log10(4e17)
for L in (512, 1024, 2048):
    log10_seconds = math.log10(t0) + (L / 2) * math.log10(2)
    print(
        f"{L:6d} "
        f"{'10^' + format(log10_seconds, '.1f'):>12} "
        f"{'10^' + format(log10_seconds - log10_year, '.1f'):>12} "
        f"{'10^' + format(log10_seconds - log10_universe, '.1f'):>22}"
    )

print(
    "\n1. Отношения к предыдущей строке: "
    + ", ".join(f"{ratio:.2f}" for ratio in ratios)
    + ".\n"
    "   Они держатся около 4, как и предсказывает рост √n при L → L+4.\n"
    "   Разброс дают три случайных модуля и то, что длина pq бывает L или L−1.\n"
    "\n2. Число 56401910421778813463 длиннее всех строк таблицы, но его наименьший\n"
    "   простой делитель равен 11: цикл smallest_factor останавливается сразу.\n"
    "   В RSA нельзя брать маленький простой множитель и нельзя, чтобы один из p, q\n"
    "   был гораздо короче другого. Оба множителя берут одной большой длины,\n"
    "   тогда наименьший делитель сам имеет длину около L/2.\n"
    "\n3. Лучшие алгоритмы разложения быстрее любой экспоненты 2^{cL}, но не полиномиальны по L.\n"
    "   Полиномиальный алгоритм остаётся посильным при росте L. Суперполиномиальный\n"
    "   при достаточной длине модуля выходит за любой практический ресурс.\n"
    "   Ускорение конкретной программы меняет только множитель перед оценкой,\n"
    "   а выбор длины ключа опирается именно на этот рост."
)


Гипотеза. У модуля n = pq с простыми по L/2 бит наименьший делитель около √n.
Перебор в smallest_factor доходит примерно до 2^{L/2}.
Плюс 4 бита умножают эту границу на 4, значит и время должно вырасти примерно в 4 раза.

   L     медиана, с    отношение
  32         0.0003            —
  36         0.0017         5.01
  40         0.0059         3.46
  44         0.0239         4.03
  48         0.0960         4.02
  52         0.3682         3.84

t0 по строке L = 52: 5.487e-09 с.
     L      секунды         годы     возрасты Вселенной
   512      10^68.8      10^61.3                10^51.2
  1024     10^145.9     10^138.4               10^128.3
  2048     10^300.0     10^292.5               10^282.4

1. Отношения к предыдущей строке: 5.01, 3.46, 4.03, 4.02, 3.84.
   Они держатся около 4, как и предсказывает рост √n при L → L+4.
   Разброс дают три случайных модуля и то, что длина pq бывает L или L−1.

2. Число 56401910421778813463 длиннее всех строк таблицы, но его наименьший
   про

## 3.2. Мультипликативные функции

Разложение (3.1) полезно не только само по себе: по нему быстро вычисляются целые семейства арифметических функций. Нам понадобится одна из них — функция Эйлера, — но сначала посмотрим на общий приём на двух более простых примерах.

### Определение 3.1. Мультипликативная функция

Функция $f:\mathbb N\to\mathbb Z$ называется **мультипликативной**, если $f(1)=1$ и

$$
f(mn)=f(m)\,f(n)\qquad\text{для всех взаимно простых } m,n.
\tag{3.2}
$$

Если $n$ имеет разложение (3.1), то множители $p_i^{e_i}$ попарно взаимно просты, и из (3.2) следует

$$
f(n)=f\bigl(p_1^{e_1}\bigr)\,f\bigl(p_2^{e_2}\bigr)\cdots f\bigl(p_k^{e_k}\bigr).
\tag{3.3}
$$

Значит, мультипликативную функцию достаточно знать на степенях простых чисел. Именно это и делает функция `mult_function`: по правилу `f_pp(p, e)` для $f(p^e)$ она строит функцию $f(n)$ для любого $n$. Подробнее о мультипликативных функциях, $\tau$ и $\sigma$ — [3, §1.4.1–1.4.2, с. 63–69], [4, §7.1–7.2, с. 239–253]; реализация через разложение пробным делением — [6, §3.3–3.4, с. 64–69].

In [7]:
# Мультипликативную f достаточно знать на степенях простых: формула (3.3).
# f_pp(p, e) задаёт f(p^e). Для n = p1^e1 * ... * pk^ek ответ — произведение этих значений.
# result начинает с 1, и для n = 1 цикл по пустому разложению ничего не умножает: f(1) = 1.
def mult_function(f_pp):
    '''
    По функции f_pp(p, e), задающей значения f(p**e),
    возвращает мультипликативную функцию f(n), вычисляемую по формуле (3.3).
    '''
    def f(n):
        result = 1
        for p, e in decompose(n).items():
            # Множители p^e попарно взаимно просты, поэтому f(n) распадается в произведение.
            result *= f_pp(p, e)
        return result
    return f

### Два примера: число и сумма делителей

Обозначим

$$
\tau(n)=\#\{d\ge1:\ d\mid n\},\qquad \sigma(n)=\sum_{d\mid n,\ d\ge1} d.
\tag{3.4}
$$

Обе функции мультипликативны, поэтому их достаточно вычислить для степеней простых. Делители числа $p^e$ — это $1,p,\ldots,p^e$: их $e+1$ штук, а их сумма — геометрическая прогрессия:

$$
\tau(p^e)=e+1,\qquad \sigma(p^e)=1+p+\cdots+p^e=\frac{p^{e+1}-1}{p-1}.
\tag{3.5}
$$

<div style="background-color: #141515; padding: 20px; border-radius: 10px; border-left: 4px solid #007cba;">

### Задание 3.3. $\tau$ и $\sigma$ через разложение

1. Напишите `tau_pp(p, e)` и `sigma_pp(p, e)` по формулам (3.5) и получите `tau = mult_function(tau_pp)`, `sigma = mult_function(sigma_pp)`. В `sigma_pp` используйте целочисленное деление `//`.
2. Напишите «наивные» версии `tau_naive(n)` и `sigma_naive(n)`, которые перебирают все $d$ от $1$ до $n$.
3. Проверьте совпадение двух версий для всех $1\le n\le 2000$.

```python
print(tau(100), sigma(28), sigma(1))
```

Ожидаемый вывод: `9 56 1`.

</div>

In [8]:
# Ваше решение:

# Делители p^e — это 1, p, p^2, ..., p^e. Их ровно e + 1 штук.
# Сумма этих делителей — геометрическая прогрессия (p^{e+1} - 1) / (p - 1).
# Деление целое: сумма делителей всегда целая, «//» не теряет остаток.

def tau_pp(p, e):
    return e + 1


def sigma_pp(p, e):
    return (p ** (e + 1) - 1) // (p - 1)


# tau(n) — число делителей, sigma(n) — их сумма. Обе мультипликативны,
# поэтому на общем n их собирает mult_function по формуле (3.3).
tau = mult_function(tau_pp)
sigma = mult_function(sigma_pp)


# Наивная версия не знает разложения: смотрит каждый d от 1 до n
# и проверяет, делится ли n на d. Это определение (3.4) буквально.
def tau_naive(n):
    return sum(1 for d in range(1, n + 1) if n % d == 0)


def sigma_naive(n):
    return sum(d for d in range(1, n + 1) if n % d == 0)


print(tau(100), sigma(28), sigma(1))
assert (tau(100), sigma(28), sigma(1)) == (9, 56, 1)

# На 1..2000 обе пары функций обязаны совпасть. Расхождение значило бы
# ошибку либо в формуле (3.5), либо в сборке через mult_function.
mismatches = [
    n for n in range(1, 2001) if tau(n) != tau_naive(n) or sigma(n) != sigma_naive(n)
]
print(f"tau и sigma совпали с наивными версиями на 1..2000, расхождений: {len(mismatches)}")
assert not mismatches


9 56 1
tau и sigma совпали с наивными версиями на 1..2000, расхождений: 0


<div style="background-color: #212426; padding: 20px; border-radius: 10px; border-left: 4px solid #007cba;">

### Задание 3.4. Где нужна взаимная простота

1. Проверьте равенство $\sigma(mn)=\sigma(m)\sigma(n)$ для **всех** пар взаимно простых $1\le m,n\le100$.
2. Найдите пару $m,n\le10$, не взаимно простых, для которой равенство нарушается. Объясните, какие делители числа $mn$ «теряются» или «считаются дважды».
3. *(Дополнительно.)* Пусть $f(n)$ — число делителей $n$, не делящихся на $4$. Например, у $60$ таких делителей 8 ($1,2,3,5,6,10,15,30$), у $144$ — шесть. Найдите $f(p^e)$ для $p=2$ и для нечётного $p$ и реализуйте $f$ через `mult_function`.

</div>

In [9]:
# Ваше решение:

# 1. σ мультипликативна только на взаимно простых аргументах.
# Перебираем все пары 1..100 и оставляем те, где gcd = 1, но равенство сломано.
# Такой список должен оказаться пустым.

failures = [
    (m, n)
    for m in range(1, 101)
    for n in range(1, 101)
    if gcd(m, n) == 1 and sigma(m * n) != sigma(m) * sigma(n)
]
print(f"1. Взаимно простых пар 1..100 проверено, нарушений σ(mn)=σ(m)σ(n): {len(failures)}.")
assert not failures

# 2. Ищем первую пару до 10, которая не взаимно проста и на которой равенство уже ложно.
# next берёт первый подходящий элемент генератора.
example = next(
    (m, n)
    for m in range(1, 11)
    for n in range(1, 11)
    if gcd(m, n) != 1 and sigma(m * n) != sigma(m) * sigma(n)
)
m, n = example
print(f"2. Пара m = {m}, n = {n}, gcd = {gcd(m, n)}.")
print(f"   σ({m * n}) = {sigma(m * n)}, σ({m})σ({n}) = {sigma(m) * sigma(n)}.")

div_m = [d for d in range(1, m + 1) if m % d == 0]
div_n = [d for d in range(1, n + 1) if n % d == 0]
# σ(m)σ(n) суммирует произведения «делитель m × делитель n».
# Это равно σ(mn) только когда каждая пара даёт свой делитель mn ровно один раз.
# Словарь products собирает, какие пары схлопнулись в одно и то же произведение.
products = {}
for left in div_m:
    for right in div_n:
        products.setdefault(left * right, []).append((left, right))
collisions = {value: pairs for value, pairs in products.items() if len(pairs) > 1}
print(f"   Одинаковые произведения пар делителей: {collisions}.")
print(
    "   Равенство σ(mn)=σ(m)σ(n) — это биекция между парами делителей и делителями mn.\n"
    "   Она есть только при gcd(m, n)=1. Здесь один и тот же делитель получается\n"
    "   двумя парами, поэтому в произведении σ(m)σ(n) он посчитан дважды."
)


# 3. f(n) — число делителей n, не делящихся на 4. На степенях простых:
# у нечётного p все делители нечётны, на 4 не делится ни один, поэтому f(p^e) = e + 1;
# у p = 2 делители 1, 2, 4, ..., 2^e, и на 4 не делятся только 1 и 2, поэтому f(2^e) = 2.
def f_pp(p, e):
    """Число делителей p^e, не делящихся на 4."""
    if p == 2:
        # Делители: 1, 2, 4, ..., 2^e. На 4 не делятся только 1 и 2.
        return 2
    return e + 1


f = mult_function(f_pp)


# Прямой подсчёт для сверки: делитель d проходит, если n делится на d и d не делится на 4.
def f_naive(n):
    return sum(1 for d in range(1, n + 1) if n % d == 0 and d % 4 != 0)


assert f(60) == 8 and f(144) == 6
assert all(f(n) == f_naive(n) for n in range(1, 501))
print(f"3. f(60) = {f(60)}, f(144) = {f(144)}. Для нечётного p все делители нечётны, f(p^e)=e+1.")
print("   Для p=2 и e>=1 остаются только 1 и 2, поэтому f(2^e)=2.")


1. Взаимно простых пар 1..100 проверено, нарушений σ(mn)=σ(m)σ(n): 0.
2. Пара m = 2, n = 2, gcd = 2.
   σ(4) = 7, σ(2)σ(2) = 9.
   Одинаковые произведения пар делителей: {2: [(1, 2), (2, 1)]}.
   Равенство σ(mn)=σ(m)σ(n) — это биекция между парами делителей и делителями mn.
   Она есть только при gcd(m, n)=1. Здесь один и тот же делитель получается
   двумя парами, поэтому в произведении σ(m)σ(n) он посчитан дважды.
3. f(60) = 8, f(144) = 6. Для нечётного p все делители нечётны, f(p^e)=e+1.
   Для p=2 и e>=1 остаются только 1 и 2, поэтому f(2^e)=2.


<div style="background-color: #1e1d19; padding: 20px; border-radius: 10px; border-left: 4px solid #d39e00;">

### 🔬 Практика 3.2. По определению или по разложению

**Гипотеза.** Что быстрее для $n\approx10^7$: `sigma_naive(n)` или `sigma(n)`? Во сколько раз, по вашей оценке?

Замерьте время обеих версий для нескольких $n$ и заполните таблицу.

| $n$ | `sigma_naive`, с | `sigma`, с |
|---:|---:|---:|
| $10^4$ | | |
| $10^5$ | | |
| $10^6$ | | |
| $10^7$ | | |
| $2^{60}$ | — | |
| $p\cdot q$, $p,q$ по 26 бит | — | |

**Вопросы.**

1. Сколько шагов делает каждая версия как функция от $n$, а затем от $L$?
2. Почему `sigma` мгновенна на $2^{60}$ и заметно медленнее на 52-битном $pq$? Где на самом деле тратится время формулы (3.3)?
3. Сформулируйте вывод одной фразой: формула (3.3) быстра, **если** ...

</div>

In [10]:
# Выполните практику здесь:

print(
    "Гипотеза. sigma_naive для n≈10^7 делает около 10^7 шагов.\n"
    "sigma почти всё время тратит на разложение. У случайного n≈10^7 маленький\n"
    "простой делитель находится быстро, поэтому sigma должна быть в сотни или тысячи раз быстрее."
)


# Короткий вызов короче шума таймера. Сначала один прогон.
# Если он уже длиннее min_seconds, повторять нельзя: наивный σ(10^7) и так тяжёлый.
# Если вызов короткий, повторяем его столько раз, чтобы суммарно набрать min_seconds,
# и делим время на число повторов.
def time_call(fn, arg, min_seconds=0.05):
    started = time.perf_counter()
    fn(arg)
    elapsed = time.perf_counter() - started
    if elapsed >= min_seconds:
        return elapsed
    repeats = max(1, int(min_seconds / max(elapsed, 1e-6)))
    started = time.perf_counter()
    for _ in range(repeats):
        fn(arg)
    return (time.perf_counter() - started) / repeats


print(f"\n{'n':>14} {'sigma_naive, с':>16} {'sigma, с':>14}")
for n in (10**4, 10**5, 10**6, 10**7):
    naive_time = time_call(sigma_naive, n)
    fast_time = time_call(sigma, n)
    print(f"{n:14d} {naive_time:16.6f} {fast_time:14.6f}")

# 2^60 наивная функция не считает: у неё 2^60 шагов. Через разложение это мгновенно,
# потому что наименьший делитель равен 2 и цикл останавливается на первом кандидате.
power_of_two = 2**60
power_time = time_call(sigma, power_of_two)
print(f"{'2**60':>14} {'—':>16} {power_time:14.6f}")

# pq из двух 26-битных простых — худший случай пробного деления на этой длине:
# меньший делитель сам около 26 бит, и формула (3.3) почти ничего не стоит рядом с поиском.
# min_seconds=0: один прогон уже доли секунды, повторять его не нужно.
left, right = random_prime(26), random_prime(26)
semiprime = left * right
semiprime_time = time_call(sigma, semiprime, min_seconds=0)
print(f"{'pq, 26 бит':>14} {'—':>16} {semiprime_time:14.6f}   n = {semiprime}")

print(
    "\n1. sigma_naive перебирает d от 1 до n: Θ(n) = Θ(2^L) шагов.\n"
    "   sigma сначала ищет разложение пробным делением. В худшем случае это Θ(√n) = Θ(2^{L/2})\n"
    "   шагов, а сама формула (3.3) после этого делает O(ω(n)) = O(L) умножений.\n"
    "\n2. У 2^60 наименьший делитель равен 2, и пробное деление заканчивается на первом кандидате.\n"
    "   У 52-битного pq наименьший делитель сам около 26 бит. Время сидит в smallest_factor,\n"
    "   а не в умножении значений σ(p^e) по формуле (3.3).\n"
    "\n3. Формула (3.3) быстра, если каноническое разложение n уже известно."
)


Гипотеза. sigma_naive для n≈10^7 делает около 10^7 шагов.
sigma почти всё время тратит на разложение. У случайного n≈10^7 маленький
простой делитель находится быстро, поэтому sigma должна быть в сотни или тысячи раз быстрее.

             n   sigma_naive, с       sigma, с
         10000         0.000174       0.000001
        100000         0.001776       0.000001
       1000000         0.017250       0.000002
      10000000         0.173735       0.000002
         2**60                —       0.000005
    pq, 26 бит                —       0.449134   n = 2538798906907399

1. sigma_naive перебирает d от 1 до n: Θ(n) = Θ(2^L) шагов.
   sigma сначала ищет разложение пробным делением. В худшем случае это Θ(√n) = Θ(2^{L/2})
   шагов, а сама формула (3.3) после этого делает O(ω(n)) = O(L) умножений.

2. У 2^60 наименьший делитель равен 2, и пробное деление заканчивается на первом кандидате.
   У 52-битного pq наименьший делитель сам около 26 бит. Время сидит в smallest_factor,
   а не в умно

<div style="border-left: 3px solid #666; padding-left: 18px; margin: 18px 0;">

**Взгляд вперёд.** Число $n$ называется совершенным, если $\sigma(n)=2n$ (например, $6$ и $28$). Чётные совершенные числа связаны с простыми числами Мерсенна $2^p-1$, для которых есть очень быстрый тест простоты Люка—Лемера. (Теорема Евклида—Эйлера, [3, теорема 1.4.10, с. 72–73], [4, теорема 7.10, с. 256]; тест Люка—Лемера — [4, теорема 7.13, с. 260].) Эта тема — в ноутбуке 4, где речь пойдёт о проверке простоты и генерации больших простых чисел.

</div>

<div style="background-color: #171819; padding: 20px; border-radius: 10px; border-left: 4px solid #007cba;">

### Задание 3.5. Совершенные, избыточные и дружественные числа

Сравним $\sigma(n)$ с $2n$. Число $n$ называется **совершенным**, если $\sigma(n)=2n$, **недостаточным**, если $\sigma(n)<2n$, и **избыточным**, если $\sigma(n)>2n$. Иначе говоря, сравнивается сумма собственных делителей $\sigma(n)-n$ с самим $n$. Два различных числа $m,n$ называются **дружественными**, если $\sigma(m)=\sigma(n)=m+n$: сумма собственных делителей каждого из них равна другому числу.

**Гипотеза.** До вычислений предскажите: каких чисел до $10^4$ больше — недостаточных или избыточных? Бывают ли нечётные избыточные числа?

1. Напишите функцию `is_perfect(n)`, которая определяет, является ли число совершенным; используйте `sigma` из задания 3.3. Найдите все совершенные числа $n\le10^4$. Проверьте:

```python
print(is_perfect(10), is_perfect(28))
```

Ожидаемый вывод: `False True`.

Запишите каждое найденное число в виде $2^{k-1}(2^k-1)$. Каким оказывается множитель $2^k-1$?

2. Разбейте числа $1\le n\le10^4$ на три класса и заполните таблицу. Найдите шесть наименьших избыточных чисел и наименьшее нечётное избыточное число.

| класс | недостаточные | совершенные | избыточные |
|---|---:|---:|---:|
| количество $n\le10^4$ | | | |

3. Найдите все пары дружественных чисел $m<n<10^4$. Для самопроверки: наименьшая пара — $220$ и $284$.

**Вопросы.**

1. Почему любая степень простого $p^e$ — недостаточное число? Воспользуйтесь формулой (3.5).
2. Совпал ли ответ с гипотезой? Какая доля чисел до $10^4$ избыточна?

Определения и упражнения — [4, §7.3, с. 256, 266–269]: упражнения 1, 5–6 и 15, Computations and Explorations 7, Programming Projects 1.

</div>

In [11]:
# Ваше решение:

# n совершенно, когда сумма всех делителей равна 2n.
# Сумма собственных делителей (без самого n) тогда равна n.
# σ(n) < 2n — недостаточное, σ(n) > 2n — избыточное.

def is_perfect(n):
    return sigma(n) == 2 * n


print(is_perfect(10), is_perfect(28))
assert is_perfect(10) is False and is_perfect(28) is True

print(
    "Гипотеза. Недостаточных больше: всякая степень простого недостаточна, а простых много.\n"
    "Нечётные избыточные числа бывают, но они редки и первое из них, если оно есть, больше 100."
)

# σ считаем один раз на каждое n и дальше только сравниваем с 2n.
# Повторный вызов sigma в трёх проходах делал бы одно и то же разложение трижды.
values = {n: sigma(n) for n in range(1, 10_001)}
perfect = [n for n, total in values.items() if total == 2 * n]
deficient = sum(1 for n, total in values.items() if total < 2 * n)
abundant_numbers = [n for n, total in values.items() if total > 2 * n]
odd_abundant = next(n for n in abundant_numbers if n % 2 == 1)

print("Совершенные n <= 10^4:", perfect)
for n in perfect:
    # Чётное совершенное число имеет вид 2^{k-1} * (2^k - 1), где 2^k - 1 простое.
    # Снимаем все двойки: exponent = k - 1, оставшееся mersenne и есть 2^k - 1.
    exponent = 0
    mersenne = n
    while mersenne % 2 == 0:
        mersenne //= 2
        exponent += 1
    k = exponent + 1
    print(
        f"  {n} = 2^{exponent} * (2^{k} - 1) = 2^{exponent} * {mersenne}; "
        f"множитель 2^{k}-1 простой: {is_prime_fast(mersenne)}"
    )

print(f"\nнедостаточные {deficient}")
print(f"совершенные   {len(perfect)}")
print(f"избыточные    {len(abundant_numbers)}")
print("шесть наименьших избыточных:", abundant_numbers[:6])
print("наименьшее нечётное избыточное:", odd_abundant)

# Дружественные m < n: сумма собственных делителей m равна n, и наоборот.
# Сумма собственных делителей — это σ(m) - m. Партнёр должен тоже лежать до 10^4.
pairs = []
for m, total in values.items():
    # total = σ(m). Условие σ(m) = m + n записывается как n = σ(m) - m.
    # Вторая проверка values[partner] - partner == m — это симметрия.
    partner = total - m
    if m < partner < 10_001 and values[partner] - partner == m:
        pairs.append((m, partner))
print("дружественные пары m < n < 10^4:", pairs)
assert pairs[0] == (220, 284)

print(
    "\n1. По (3.5) σ(p^e) = (p^{e+1}-1)/(p-1).\n"
    "   2 p^e (p-1) - (p^{e+1}-1) = p^e (p-2) + 1 >= 1,\n"
    "   поэтому σ(p^e) < 2 p^e. Всякая степень простого недостаточна.\n"
    f"\n2. Недостаточных {deficient} из 10000, избыточных {len(abundant_numbers)} "
    f"(доля {len(abundant_numbers) / 10_000:.4f}).\n"
    "   Недостаточных больше, гипотеза совпала.\n"
    f"   Нечётные избыточные есть: наименьшее равно {odd_abundant}."
)


False True
Гипотеза. Недостаточных больше: всякая степень простого недостаточна, а простых много.
Нечётные избыточные числа бывают, но они редки и первое из них, если оно есть, больше 100.
Совершенные n <= 10^4: [6, 28, 496, 8128]
  6 = 2^1 * (2^2 - 1) = 2^1 * 3; множитель 2^2-1 простой: True
  28 = 2^2 * (2^3 - 1) = 2^2 * 7; множитель 2^3-1 простой: True
  496 = 2^4 * (2^5 - 1) = 2^4 * 31; множитель 2^5-1 простой: True
  8128 = 2^6 * (2^7 - 1) = 2^6 * 127; множитель 2^7-1 простой: True

недостаточные 7508
совершенные   4
избыточные    2488
шесть наименьших избыточных: [12, 18, 20, 24, 30, 36]
наименьшее нечётное избыточное: 945
дружественные пары m < n < 10^4: [(220, 284), (1184, 1210), (2620, 2924), (5020, 5564), (6232, 6368)]

1. По (3.5) σ(p^e) = (p^{e+1}-1)/(p-1).
   2 p^e (p-1) - (p^{e+1}-1) = p^e (p-2) + 1 >= 1,
   поэтому σ(p^e) < 2 p^e. Всякая степень простого недостаточна.

2. Недостаточных 7508 из 10000, избыточных 2488 (доля 0.2488).
   Недостаточных больше, гипотеза совпал

## 3.3. Функция Эйлера и группа $(\mathbb Z/n\mathbb Z)^*$

Классы вычетов $\mathbb Z/n\mathbb Z$ и их арифметика определены в ноутбуке 1 (определения 1.2–1.4, формулы (1.9)–(1.12)). Там же (формула (1.23)) доказано, что класс $[a]_n$ обратим тогда и только тогда, когда $\gcd(a,n)=1$. Это свойство не зависит от выбора представителя: если $a\equiv b\pmod n$, то $\gcd(a,n)=\gcd(b,n)$.

### Определение 3.2. Функция Эйлера

Для $n\ge1$

$$
\varphi(n)=\#\{\,k:\ 1\le k\le n,\ \gcd(k,n)=1\,\}.
\tag{3.6}
$$

В частности, $\varphi(1)=1$, и $\varphi(p)=p-1$ для простого $p$.

### Определение 3.3. Группа обратимых классов

Множество обратимых классов по модулю $n$ обозначается

$$
(\mathbb Z/n\mathbb Z)^*=\{\,[a]_n:\ \gcd(a,n)=1\,\}.
\tag{3.7}
$$

Произведение обратимых классов обратимо, обратный к обратимому классу тоже обратим, поэтому $(\mathbb Z/n\mathbb Z)^*$ — группа по умножению. Её порядок (число элементов) равен $\varphi(n)$. Определения $\varphi$ и группы $(\mathbb Z/n\mathbb Z)^*$ — [1, §1.3, с. 22], [3, §1.4.4, с. 78–79]; мультипликативность и формула (3.8) — [4, теоремы 7.4–7.5, с. 241–242], [5, предложение 8.6 и теорема 8.8, с. 194–195]. Два способа вычисления $\varphi$ — подсчётом НОД и через разложение — сравниваются в [6, §3.1, с. 52–57]. Набор из $\varphi(n)$ чисел, взятых по одному из каждого обратимого класса, называется **приведённой системой вычетов** по модулю $n$.

Например, $(\mathbb Z/9\mathbb Z)^*=\{1,2,4,5,7,8\}$ и $\varphi(9)=6$.

### Как вычислять $\varphi(n)$

Функция Эйлера мультипликативна. Это прямо следует из КТО (ноутбук 1, раздел 1.7): при $\gcd(m,n)=1$ класс по модулю $mn$ — это пара остатков по модулям $m$ и $n$, а взаимная простота с $mn$ означает взаимную простоту с $m$ и с $n$ одновременно. Значит, обратимых пар ровно $\varphi(m)\varphi(n)$.

На степени простого из чисел $1,\ldots,p^e$ необратимы ровно кратные $p$, их $p^{e-1}$. Вместе с (3.3) получаем

$$
\varphi(p^e)=p^e-p^{e-1},\qquad \varphi(n)=n\prod_{p\mid n}\Bigl(1-\frac1p\Bigr).
\tag{3.8}
$$

<div style="background-color: #1b1c1d; padding: 20px; border-radius: 10px; border-left: 4px solid #007cba;">

### Задание 3.6. Функция Эйлера двумя способами

1. Напишите `euler_fun(n)` прямо по определению (3.6), используя `gcd` из `crypto_math`.
2. Напишите `euler_pp(p, e)` по формуле (3.8) и получите `euler_phi = mult_function(euler_pp)`.
3. Проверьте совпадение двух версий для $1\le n\le2000$.

```python
print(euler_fun(1), euler_phi(1), euler_phi(9), euler_phi(1125))
```

Ожидаемый вывод: `1 1 6 600`.

</div>

In [12]:
# Ваше решение:

# euler_fun считает φ(n) по определению (3.6): сколько k от 1 до n взаимно просто с n.
# euler_pp — формула на степени простого: из p^e чисел кратны p ровно p^{e-1} штук,
# поэтому обратимых p^e - p^{e-1}. Общее n собирает mult_function.

def euler_fun(n):
    return sum(1 for k in range(1, n + 1) if gcd(k, n) == 1)


def euler_pp(p, e):
    return p**e - p ** (e - 1)


euler_phi = mult_function(euler_pp)

print(euler_fun(1), euler_phi(1), euler_phi(9), euler_phi(1125))
assert (euler_fun(1), euler_phi(1), euler_phi(9), euler_phi(1125)) == (1, 1, 6, 600)

# Два способа обязаны совпасть. Для n = 1 оба дают 1: определение считает k = 1,
# а mult_function на пустом разложении оставляет результат 1.
mismatches = [n for n in range(1, 2001) if euler_fun(n) != euler_phi(n)]
print(f"euler_fun и euler_phi совпали на 1..2000, расхождений: {len(mismatches)}")
assert not mismatches


1 1 6 600
euler_fun и euler_phi совпали на 1..2000, расхождений: 0


<div style="background-color: #465057; padding: 20px; border-radius: 10px; border-left: 4px solid #007cba;">

### Задание 3.7. Разминка: сравнимые пары

Среди чисел $135,\ -14,\ 181,\ -225,\ 167,\ 452$ найдите все пары, сравнимые между собой по модулю $15$. Сначала вручную, затем программно.

</div>

In [13]:
# Ваше решение:

print("Вручную, остатки по модулю 15:")
print("  135 = 9·15 + 0")
print("  -14 = (-1)·15 + 1")
print("  181 = 12·15 + 1")
print("  -225 = (-15)·15 + 0")
print("  167 = 11·15 + 2")
print("  452 = 30·15 + 2")
print("Пары: (135, -225) по остатку 0, (-14, 181) по остатку 1, (167, 452) по остатку 2.")

# Программная проверка ручного счёта. В Python a % 15 для отрицательного a
# уже даёт остаток от 0 до 14, тот же класс вычетов. Числа с одним остатком сравнимы.
numbers = [135, -14, 181, -225, 167, 452]
groups = {}
for number in numbers:
    groups.setdefault(number % 15, []).append(number)

print("\nПрограммно:")
for residue, group in sorted(groups.items()):
    print(f"  остаток {residue}: {group}")

assert groups[0] == [135, -225]
assert groups[1] == [-14, 181]
assert groups[2] == [167, 452]


Вручную, остатки по модулю 15:
  135 = 9·15 + 0
  -14 = (-1)·15 + 1
  181 = 12·15 + 1
  -225 = (-15)·15 + 0
  167 = 11·15 + 2
  452 = 30·15 + 2
Пары: (135, -225) по остатку 0, (-14, 181) по остатку 1, (167, 452) по остатку 2.

Программно:
  остаток 0: [135, -225]
  остаток 1: [-14, 181]
  остаток 2: [167, 452]


<div style="background-color: #2a2d2f; padding: 20px; border-radius: 10px; border-left: 4px solid #007cba;">

### Задание 3.8. Разминка: приведённая система

Образуют ли числа $-7,\ -21,\ 45,\ 271$ приведённую систему вычетов по модулю $8$? Сформулируйте, какие два условия нужно проверить.

</div>

In [14]:
# Ваше решение:

# Приведённая система вычетов по модулю n — это φ(n) чисел, которые
# 1) каждое взаимно просто с n, 2) попарно несравнимы по модулю n.
# Остаток a % 8 приводит отрицательные к диапазону 0..7, сравнимость от этого не меняется.

candidates = [-7, -21, 45, 271]
modulus = 8
residues = [number % modulus for number in candidates]
coprime = [gcd(number, modulus) == 1 for number in candidates]

print("Остатки по модулю 8:", residues)
print("Каждое взаимно просто с 8:", coprime)
# Множество короче списка, если два числа дали один и тот же остаток, то есть сравнимы.
print("Все остатки различны:", len(set(residues)) == len(residues))
print("Чисел столько же, сколько φ(8) =", euler_phi(modulus))
print(
    "\nДва условия приведённой системы: каждое число взаимно просто с модулем,\n"
    "и числа попарно несравнимы. При количестве φ(n) этих условий достаточно.\n"
    "Оба выполнены, остатки 1, 3, 5, 7 — это вся группа (Z/8Z)*.\n"
    "Да, -7, -21, 45, 271 образуют приведённую систему вычетов по модулю 8."
)
assert residues == [1, 3, 5, 7]
assert all(coprime)
assert len(set(residues)) == euler_phi(modulus)


Остатки по модулю 8: [1, 3, 5, 7]
Каждое взаимно просто с 8: [True, True, True, True]
Все остатки различны: True
Чисел столько же, сколько φ(8) = 4

Два условия приведённой системы: каждое число взаимно просто с модулем,
и числа попарно несравнимы. При количестве φ(n) этих условий достаточно.
Оба выполнены, остатки 1, 3, 5, 7 — это вся группа (Z/8Z)*.
Да, -7, -21, 45, 271 образуют приведённую систему вычетов по модулю 8.


<div style="background-color: #191a1b; padding: 20px; border-radius: 10px; border-left: 4px solid #007cba;">

### Задание 3.9. Класс `ZnZ`

Дальше удобно работать с группой как с объектом. Допишите класс `ZnZ`, представляющий группу $(\mathbb Z/n\mathbb Z)^*$:

- `elems()` — отсортированный список наименьших положительных представителей обратимых классов;
- `group_order()` — порядок группы, то есть $\varphi(n)$;
- `mod()` — модуль $n$;
- `__repr__` — печатает список элементов.

В следующих разделах класс будет пополняться новыми методами. Чтобы не путать понятия, метод для порядка **группы** называется `group_order`, а порядок **элемента** появится позже как `multiplicative_order`.

```python
G = ZnZ(24)
print(G)
print(G.group_order(), euler_phi(24))
```

Ожидаемый вывод:

```text
[1, 5, 7, 11, 13, 17, 19, 23]
8 8
```

</div>

In [15]:
# Ваше решение:
# Объект — группа обратимых классов (Z/nZ)* . Элемент храним наименьшим
# положительным представителем: 1, 2, ..., n-1, взаимно простым с n.

class ZnZ:
    """Мультипликативная группа (Z/nZ)* для n >= 2.

    Список элементов строится при первом вызове elems(), а не в __init__.
    Иначе само создание ZnZ(p) для большого простого p стало бы перебором.
    """

    def __init__(self, n):
        if n < 2:
            raise ValueError("n должно быть не меньше 2")
        # Список элементов не строим в __init__. Для большого простого p это был бы
        # перебор всех 1..p-1 ещё до того, как у группы что-то спросили.
        # None значит «ещё не считали». Список появится при первом elems().
        self.n = n
        self._elems = None

    # Модуль группы. Отдельный метод, чтобы не читать поле self.n снаружи.
    def mod(self):
        return self.n

    def elems(self):
        if self._elems is None:
            # Обратимые классы — те, у которых gcd(a, n) = 1. Это критерий из ноутбука 1.
            # range до n не включает n: n ≡ 0 и с модулем не взаимно прост.
            self._elems = [a for a in range(1, self.n) if gcd(a, self.n) == 1]
        return self._elems

    # Порядок группы равен φ(n). Считаем формулой, а не len(elems()):
    # иначе вопрос «сколько элементов» для большого p снова строил бы весь список.
    def group_order(self):
        return euler_phi(self.n)

    # Печать объекта — сам список представителей, как в примере задания.
    def __repr__(self):
        return str(self.elems())


G = ZnZ(24)
print(G)
print(G.group_order(), euler_phi(24))
assert repr(G) == "[1, 5, 7, 11, 13, 17, 19, 23]"
assert G.group_order() == euler_phi(24) == 8


[1, 5, 7, 11, 13, 17, 19, 23]
8 8


<div style="background-color: #0f0f0e; padding: 20px; border-radius: 10px; border-left: 4px solid #d39e00;">

### 🔬 Практика 3.3. Почему $\varphi(n)$ — секрет RSA

Для простого $p$ значение $\varphi(p)=p-1$ известно сразу. Для $n=pq$ по (3.8)

$$
\varphi(n)=(p-1)(q-1)=n-(p+q)+1.
$$

Тот, кто знает $p$ и $q$, вычисляет $\varphi(n)$ мгновенно. Покажем, что верно и обратное: знание $\varphi(n)$ так же «опасно», как знание разложения.

1. Выразите $p+q$ через $n$ и $\varphi(n)$. Объясните, почему $p$ и $q$ — корни квадратного уравнения
   $$
   x^2-(n-\varphi(n)+1)\,x+n=0.
   $$
2. Ниже дан 1024-битный модуль `n` и значение `phi_n`. Найдите $p$ и $q$, используя только целочисленную арифметику (`math.isqrt` вместо `** 0.5`), и проверьте, что `p * q == n`.
3. Можно ли было получить `phi_n` с помощью `euler_phi(n)`? Оцените время по результатам исследования 3.1.

**Вывод для RSA.** Вычислить $\varphi(n)$, не зная разложения $n$, не легче, чем разложить $n$. Поэтому $\varphi(n)$ (и вычисляемый по ней секретный показатель) хранится в тайне вместе с $p$ и $q$. Это рассуждение с числовым примером приведено в [1, замечание 3.10, с. 121–122].

</div>

In [16]:
n = 120036104219320619011406838669977830872186268069061746478144135411840450450459460454288904889161125575378032954340946031033550799930309783716480005582956628582133097169394317947310298991312488828117568961148840814051063274668298477188824027428420440635109666055292398505818276808870398936401337758281730144009
phi_n = 120036104219320619011406838669977830872186268069061746478144135411840450450459460454288904889161125575378032954340946031033550799930309783716480005582956606594684084365074843737423113020909205891386276557876637295515834854846367210331002341254270647443140920811241254640624909326995625242397641470847655634816

print(n.bit_length())

# Выполните практику здесь:
# φ(pq) = (p-1)(q-1) = pq - (p+q) + 1 = n - (p+q) + 1.
# Отсюда сумма неизвестных простых выражается через известные n и φ(n):
# p + q = n - φ(n) + 1. Вместе с pq = n это квадратное уравнение
# x^2 - (p+q) x + n = 0. Корни и есть p и q.
s = n - phi_n + 1  # p + q = n - φ(n) + 1
# Дискриминант (p+q)^2 - 4pq. Он полный квадрат, корень целый.
# math.isqrt, а не ** 0.5: у 1024-битного числа двоичная плавающая точка
# не хранит все биты и округлила бы корень.
discriminant = s * s - 4 * n
root = isqrt(discriminant)
# isqrt возвращает floor. Если дискриминант не квадрат, квадрат этого floor меньше его.
# Тогда φ(n) не соответствует разложению n = pq, и дальше делить нельзя.
assert root * root == discriminant
# Корни (s ± sqrt(D)) / 2. Меньший берём с минусом.
# Деление нацело: сумма и разность двух нечётных простых чётные.
p = (s - root) // 2
q = (s + root) // 2
assert p * q == n and p < q

print("p + q = n - φ(n) + 1 =", s)
print("p =", p)
print("q =", q)
print("p * q == n:", p * q == n)
print("длины p и q:", p.bit_length(), q.bit_length())
print(
    "По (3.8) φ(pq) = (p-1)(q-1) = n - (p+q) + 1, отсюда p+q = n - φ(n) + 1.\n"
    "Вместе с pq = n числа p и q — корни x^2 - (p+q)x + n = 0.\n"
    "Дискриминант (p+q)^2 - 4n является полным квадратом, корень берётся через math.isqrt.\n"
    "\n"
    "euler_phi(n) для этого 1024-битного модуля вызвал бы пробное деление до √n ≈ 2^{512}.\n"
    "По исследованию 3.1 это на много порядков дольше возраста Вселенной.\n"
    "Значение phi_n здесь дано вместе с секретом; из одного n его так не получить.\n"
    "Кто знает φ(n), восстанавливает p и q за полиномиальное время, поэтому φ(n)\n"
    "хранят в тайне наравне с разложением."
)


1024
p + q = n - φ(n) + 1 = 21987449012804319474209887185970403282936731292403272203518535228419821931266857821686174149793191968745244051143865193367481874773694003696287434074509194
p = 10084948239465796714750005783443521406498590427986710135150934640988740743155514001811212679760382777453724380121248974210706059353134008816038726256780817
q = 11902500773338522759459881402526881876438140864416562068367600587431081188111343819874961470032809191291519671022616219156775815420559994880248707817728377
p * q == n: True
длины p и q: 512 512
По (3.8) φ(pq) = (p-1)(q-1) = n - (p+q) + 1, отсюда p+q = n - φ(n) + 1.
Вместе с pq = n числа p и q — корни x^2 - (p+q)x + n = 0.
Дискриминант (p+q)^2 - 4n является полным квадратом, корень берётся через math.isqrt.

euler_phi(n) для этого 1024-битного модуля вызвал бы пробное деление до √n ≈ 2^{512}.
По исследованию 3.1 это на много порядков дольше возраста Вселенной.
Значение phi_n здесь дано вместе с секретом; из одного n его так не получить.
Кто знае

## 3.4. Порядок элемента. Теоремы Лагранжа, Эйлера и Ферма

### Определение 3.4. Порядок элемента

Пусть $\gcd(a,n)=1$. **Порядком элемента** $a$ по модулю $n$ называется наименьшее натуральное $k\ge1$, для которого

$$
a^k\equiv1\pmod n.
\tag{3.9}
$$

Обозначение: $\operatorname{ord}_n(a)$ или просто $\operatorname{ord}(a)$. Например, $\operatorname{ord}(1)=1$ при любом $n$.

Такое $k$ всегда существует. Степени $a,a^2,a^3,\ldots$ принимают лишь конечное число значений по модулю $n$, поэтому найдутся $i>j$ с $a^i\equiv a^j$. Так как $a$ обратим, можно сократить на $a^j$ и получить $a^{i-j}\equiv1$.

### Главное свойство порядка

$$
a^m\equiv1\pmod n\iff \operatorname{ord}(a)\mid m.
\tag{3.10}
$$

Это свойство мы будем использовать постоянно, поэтому приведём короткое объяснение. Пусть $d=\operatorname{ord}(a)$ и $m=qd+r$, где $0\le r<d$. Тогда $a^m=(a^d)^q a^r\equiv a^r$. Если $a^m\equiv1$, то $a^r\equiv1$ при $r<d$, и по минимальности $d$ остаётся $r=0$. Обратное направление очевидно.

### Теорема Лагранжа и её следствия

Из курса алгебры известна **теорема Лагранжа**: порядок подгруппы конечной группы делит порядок группы ([1, предложение 2.14, с. 74]). Степени элемента $a$ образуют подгруппу из $\operatorname{ord}(a)$ элементов (раздел 3.6), поэтому

$$
\operatorname{ord}(a)\mid\varphi(n).
\tag{3.11}
$$

<div style="border-top: 1px solid #999; border-bottom: 1px solid #999; padding: 12px 4px; margin: 18px 0;">

<strong>Теорема 3.1. Теоремы Эйлера и Ферма</strong>

Если $\gcd(a,n)=1$, то

$$
a^{\varphi(n)}\equiv1\pmod n\qquad\text{(теорема Эйлера)}.
\tag{3.12}
$$

В частности, для простого $p$ и $a\not\equiv0\pmod p$

$$
a^{p-1}\equiv1\pmod p\qquad\text{(малая теорема Ферма)},
\tag{3.13}
$$

а для любого целого $a$ выполнено $a^p\equiv a\pmod p$.

См. [1, теорема 1.25, с. 30; предложение 1.30, с. 32], [3, теоремы 1.6.15–1.6.16, с. 125–127], [4, теорема 6.14, с. 236], [5, теоремы 8.1 и 8.9, с. 190–195], [2, приложение A.2.2, с. 815]. Свойство (3.10) — [4, теорема 9.1 и следствие 9.1.1, с. 348–349], [5, теорема 11.1, с. 240].

</div>

Доказательство укладывается в одну строку: по (3.11) $\varphi(n)$ делится на $\operatorname{ord}(a)$, и по (3.10) $a^{\varphi(n)}\equiv1$.

### Что это даёт для вычислений

1. **Показатель можно сокращать по модулю $\varphi(n)$.** При $\gcd(a,n)=1$

$$
a^k\equiv a^{\,k\bmod\varphi(n)}\pmod n.
\tag{3.14}
$$

2. **Обратный элемент через степень.** Из (3.12) следует $a\cdot a^{\varphi(n)-1}\equiv1$, то есть

$$
a^{-1}\equiv a^{\varphi(n)-1}\pmod n,\qquad a^{-1}\equiv a^{p-2}\pmod p.
\tag{3.15}
$$

Это второй способ обращения, помимо `mod_inverse` из ноутбука 1. Какой из них быстрее, проверим в исследовании 3.4.

3. **Корректность RSA.** Шифрование и расшифрование в RSA — это возведение в степени $e$ и $d$ с $ed\equiv1\pmod{\varphi(n)}$. По (3.14) $m^{ed}\equiv m$. Подробно это будет разобрано в ноутбуке о RSA.

<div style="background-color: #212324; padding: 20px; border-radius: 10px; border-left: 4px solid #007cba;">

### Задание 3.10. Порядок элемента перебором

Добавьте в класс `ZnZ` метод `multiplicative_order(x)`, который вычисляет $\operatorname{ord}(x)$ последовательным умножением $x, x^2, x^3,\ldots$ до получения $1$.

```python
G = ZnZ(9)
print(G)
print({x: G.multiplicative_order(x) for x in G.elems()})
```

Ожидаемый вывод:

```text
[1, 2, 4, 5, 7, 8]
{1: 1, 2: 6, 4: 3, 5: 6, 7: 3, 8: 2}
```

Убедитесь, что все порядки делят $\varphi(9)=6$, как требует (3.11).

</div>

In [17]:
# Ваше решение:

# Порядок x — наименьшее k >= 1, для которого x^k ≡ 1 (mod n).
# Считаем степени последовательным умножением: x, x^2, x^3, ...
# Так дешевле, чем звать mod_pow на каждый показатель с нуля.

def multiplicative_order(self, x):
    # Представитель класса. Отрицательный x тоже приводится к 0..n-1.
    x %= self.n
    # У необратимого элемента степени никогда не дают 1: общий делитель с n сохраняется.
    # Порядка в группе тогда нет.
    if gcd(x, self.n) != 1:
        raise ValueError("x не обратим по модулю n")
    # power — это уже x^1. Если x ≡ 1, цикл не начинается и порядок равен 1.
    power = x
    k = 1
    while power != 1:
        # Шаг от x^k к x^{k+1}. Остаток берём сразу, чтобы число не росло как x^k без модуля.
        power = (power * x) % self.n
        k += 1
    return k


# Метод добавляется в класс после его создания: следующие задания дописывают ZnZ,
# не переписывая всё тело целиком.
ZnZ.multiplicative_order = multiplicative_order

G = ZnZ(9)
print(G)
orders = {x: G.multiplicative_order(x) for x in G.elems()}
print(orders)
assert orders == {1: 1, 2: 6, 4: 3, 5: 6, 7: 3, 8: 2}
# Теорема Лагранжа (3.11): порядок элемента делит порядок группы φ(9) = 6.
assert all(euler_phi(9) % order == 0 for order in orders.values())
print("Все порядки делят φ(9) = 6.")


[1, 2, 4, 5, 7, 8]
{1: 1, 2: 6, 4: 3, 5: 6, 7: 3, 8: 2}
Все порядки делят φ(9) = 6.


<div style="background-color: #121314; padding: 20px; border-radius: 10px; border-left: 4px solid #007cba;">

### Задание 3.11. Главное свойство порядка в действии

1. Найдите все $1\le m\le30$, для которых $4^m\equiv1\pmod 9$, и проверьте, что все они делятся на $\operatorname{ord}_9(4)$.
2. Не вычисляя $2^{10}\bmod 7$, определите с помощью (3.10), является ли $x=10$ решением сравнения $2^x\equiv1\pmod7$. Затем проверьте ответ вычислением.
3. Вычислите $3^{10^{100}}\bmod 101$ вручную с помощью (3.14) и проверьте результат через `mod_pow`.
4. Найдите все элементы $(\mathbb Z/19\mathbb Z)^*$ порядка $18$. Сколько их? Запомните это число — оно будет объяснено в разделе 3.6.

</div>

In [18]:
# Ваше решение:

G9 = ZnZ(9)
order_4 = G9.multiplicative_order(4)
# 1. По свойству (3.10) 4^m ≡ 1 (mod 9) ровно тогда, когда ord_9(4) делит m.
# Список должен совпасть с кратными порядка.
exponents = [m for m in range(1, 31) if mod_pow(4, m, 9) == 1]
print("1. 4^m ≡ 1 (mod 9) при m <= 30:", exponents)
print(f"   ord_9(4) = {order_4}. Все такие m делятся на {order_4}: {all(m % order_4 == 0 for m in exponents)}")
assert order_4 == 3
assert exponents == list(range(3, 31, 3))

# 2. Сравнение 2^10 ≡ 1 (mod 7) имеет решение только если ord_7(2) делит 10.
# Сначала порядок, и только потом проверка прямым возведением.
order_2 = ZnZ(7).multiplicative_order(2)
print(f"2. ord_7(2) = {order_2}. Делитель 10? {order_2} делит 10: {10 % order_2 == 0}.")
print("   По (3.10) сравнение 2^10 ≡ 1 (mod 7) решений не имеет.")
print("   Проверка: 2^10 mod 7 =", mod_pow(2, 10, 7))
assert order_2 == 3 and mod_pow(2, 10, 7) == 2

# 3. Модуль 101 простой, φ(101) = 100. По (3.14) показатель можно заменить
# остатком по модулю 100. 10^2 = 100 делится на 100, и любая большая степень 10 тоже.
# Остаток показателя равен 0, а 3^{100t} = (3^{100})^t ≡ 1. Это и есть 3^0.
print("3. φ(101) = 100. При показателе >= 2 число 10^k делится на 100,")
print("   потому что 10^2 = 100. Значит 10^100 ≡ 0 (mod 100).")
print("   По (3.14) 3^{10^100} ≡ 3^0 ≡ 1 (mod 101): показатель кратен φ(101),")
print("   а 3^{100} ≡ 1, поэтому и большая степень равна 1.")
assert mod_pow(3, 10**100, 101) == 1
print("   mod_pow(3, 10**100, 101) =", mod_pow(3, 10**100, 101))

# 4. Группа (Z/19Z)* имеет порядок φ(19) = 18. Элемент порядка 18 — образующий.
# В циклической группе таких ровно φ(18). Это число всплывёт снова в разделе 3.6.
generators = [a for a in ZnZ(19).elems() if ZnZ(19).multiplicative_order(a) == 18]
print("4. Элементы (Z/19Z)* порядка 18:", generators)
print(f"   Их {len(generators)}, и φ(18) = {euler_phi(18)}")
assert len(generators) == euler_phi(18) == 6


1. 4^m ≡ 1 (mod 9) при m <= 30: [3, 6, 9, 12, 15, 18, 21, 24, 27, 30]
   ord_9(4) = 3. Все такие m делятся на 3: True
2. ord_7(2) = 3. Делитель 10? 3 делит 10: False.
   По (3.10) сравнение 2^10 ≡ 1 (mod 7) решений не имеет.
   Проверка: 2^10 mod 7 = 2
3. φ(101) = 100. При показателе >= 2 число 10^k делится на 100,
   потому что 10^2 = 100. Значит 10^100 ≡ 0 (mod 100).
   По (3.14) 3^{10^100} ≡ 3^0 ≡ 1 (mod 101): показатель кратен φ(101),
   а 3^{100} ≡ 1, поэтому и большая степень равна 1.
   mod_pow(3, 10**100, 101) = 1
4. Элементы (Z/19Z)* порядка 18: [2, 3, 10, 13, 14, 15]
   Их 6, и φ(18) = 6


<div style="background-color: #252422; padding: 20px; border-radius: 10px; border-left: 4px solid #d39e00;">

### 🔬 Исследование 3.4. Обращение: Евклид или Ферма?

Обе формулы — `mod_inverse(a, p)` (ноутбук 1) и $a^{p-2}\bmod p$ по (3.15) — дают один и тот же результат, и обе работают за полиномиальное по $L$ время.

**Гипотеза.** Какой способ быстрее на 2048-битном простом $p$? Растёт или убывает отношение времён с ростом $L$? Обоснуйте прогноз оценками числа операций из ноутбука 1: расширенный алгоритм Евклида делает $O(L)$ шагов с числами длины $L$, а `mod_pow` — $O(L)$ умножений по модулю.

**Ход.** Нам нужны большие простые числа, а быстрой проверки простоты пока нет. Воспользуемся известными **простыми числами Мерсенна** $M_k=2^k-1$ (подробнее о них — в ноутбуке 4). Для каждого $k$ возьмите 20 случайных $a$, замерьте суммарное время `mod_inverse(a, p)` и `mod_pow(a, p - 2, p)` и для справки — встроенного `pow(a, p - 2, p)`.

| $L=k$ | `mod_inverse`, с | `mod_pow(a, p-2, p)`, с | отношение | встроенный `pow`, с |
|---:|---:|---:|---:|---:|
| 127 | | | | |
| 521 | | | | |
| 1279 | | | | |
| 2203 | | | | |
| 4253 | | | | |

**Вопросы.**

1. Как меняется отношение времён с ростом $L$? Согласуется ли это с оценками $O(L^2)$ для расширенного Евклида и $O(L^3)$ для возведения в степень школьным умножением?
2. Почему для простого модуля всё-таки иногда выбирают обращение по Ферма? *Подсказка:* в `mod_pow` число и порядок операций зависят только от показателя $p-2$, который известен всем, а в алгоритме Евклида — от секретного $a$.
3. Применима ли формула $a^{p-2}$, если модуль составной, например $n=pq$ в RSA? Что пришлось бы знать?

</div>

In [19]:
MERSENNE_EXPONENTS = [127, 521, 1279, 2203, 4253]   # 2**k - 1 — простые числа

# Выполните исследование здесь:
from statistics import median

print(
    "Гипотеза. Расширенный Евклид на L-битных числах стоит O(L^2) битовых операций,\n"
    "а mod_pow со школьным умножением — O(L) умножений по O(L^2), то есть O(L^3).\n"
    "Отношение mod_pow / mod_inverse должно расти с L. На 2048 битах быстрее Евклид."
)


# p = 2^k - 1 — известное простое Мерсенна. Своим is_prime_fast его не проверяем:
# для k = 4253 это пробное деление до 2^{2126}, то есть как раз то, чего мы избегаем.
# 20 случайных a, суммарное время каждого способа. В замер не входит проверка равенства.
def bench(k, samples=20):
    p = 2**k - 1
    # a от 2 до p-1: класс 0 необратим, а a = 1 слишком лёгкий частный случай.
    # Для простого p любое такое a взаимно просто с p.
    bases = [random.randrange(2, p) for _ in range(samples)]
    started = time.perf_counter()
    for a in bases:
        mod_inverse(a, p)
    inverse_time = time.perf_counter() - started
    started = time.perf_counter()
    for a in bases:
        # Обращение по Ферма: a^{p-2} ≡ a^{-1} (mod p), формула (3.15).
        mod_pow(a, p - 2, p)
    pow_time = time.perf_counter() - started
    started = time.perf_counter()
    for a in bases:
        # Встроенный pow — та же степень, но умножение длинных чисел в CPython быстрее школьного.
        # Он здесь только как ориентир скорости, учебным алгоритмом не заменяется.
        pow(a, p - 2, p)
    builtin_time = time.perf_counter() - started
    a = bases[0]
    # Три способа дают один обратимый элемент. Проверяем на одном a после замера,
    # чтобы эта лишняя степень не попала в таблицу.
    assert mod_inverse(a, p) == mod_pow(a, p - 2, p) == pow(a, p - 2, p)
    return inverse_time, pow_time, builtin_time


print(f"\n{'L':>6} {'inverse, с':>12} {'mod_pow, с':>12} {'отношение':>10} {'pow, с':>12}")
rows = []
for k in MERSENNE_EXPONENTS:
    inverse_time, pow_time, builtin_time = bench(k)
    rows.append((k, inverse_time, pow_time, pow_time / inverse_time, builtin_time))
    print(f"{k:6d} {inverse_time:12.4f} {pow_time:12.4f} {pow_time / inverse_time:10.2f} {builtin_time:12.4f}")

print(
    "\n1. Отношение mod_pow / mod_inverse по строкам: "
    + ", ".join(f"{row[0]} → {row[3]:.1f}" for row in rows)
    + ".\n"
    "   С ростом L оно увеличивается. Так и должны вести себя O(L^2) у расширенного Евклида\n"
    "   и O(L^3) у степени, если умножение школьное.\n"
    "   Рост слабее чистого множителя L: длинное умножение в CPython быстрее школьного.\n"
    "   То же видно по встроенному pow, он обгоняет учебный mod_pow.\n"
    "\n2. В обращении по Ферма показатель p-2 открыт, и последовательность умножений от a не зависит.\n"
    "   В алгоритме Евклида частные зависят от самого a. Если a секрет, по времени шагов Евклида\n"
    "   можно пытаться его восстановить, а у степени Ферма такого канала нет.\n"
    "\n3. Формула a^{p-2} использует малую теорему Ферма и требует простого модуля.\n"
    "   Для составного n = pq обратный элемент — это a^{φ(n)-1}, а φ(n)=(p-1)(q-1)\n"
    "   как раз секрет RSA. Без разложения n эту степень не построить."
)


Гипотеза. Расширенный Евклид на L-битных числах стоит O(L^2) битовых операций,
а mod_pow со школьным умножением — O(L) умножений по O(L^2), то есть O(L^3).
Отношение mod_pow / mod_inverse должно расти с L. На 2048 битах быстрее Евклид.

     L   inverse, с   mod_pow, с  отношение       pow, с
   127       0.0004       0.0007       1.60       0.0004
   521       0.0043       0.0143       3.35       0.0083
  1279       0.0344       0.1486       4.33       0.0847
  2203       0.1654       0.6883       4.16       0.3999
  4253       0.8993       4.4689       4.97       2.6130

1. Отношение mod_pow / mod_inverse по строкам: 127 → 1.6, 521 → 3.4, 1279 → 4.3, 2203 → 4.2, 4253 → 5.0.
   С ростом L оно увеличивается. Так и должны вести себя O(L^2) у расширенного Евклида
   и O(L^3) у степени, если умножение школьное.
   Рост слабее чистого множителя L: длинное умножение в CPython быстрее школьного.
   То же видно по встроенному pow, он обгоняет учебный mod_pow.

2. В обращении по Ферма показате

## 3.5. Быстрое вычисление порядка

Перебор из задания 3.10 делает до $\varphi(n)$ умножений — это экспонента по $L$. Свойство (3.11) подсказывает, что искать порядок нужно только среди делителей $t=\varphi(n)$.

**Пример.** Найдём $\operatorname{ord}_{17}(5)$. Здесь $t=\varphi(17)=16$, делители: $1,2,4,8,16$. Вычисляем $5^8\equiv16\not\equiv1\pmod{17}$. Все делители $16$, кроме самого $16$, делят $8$; если бы порядок был меньше $16$, то по (3.10) выполнялось бы $5^8\equiv1$. Значит, $\operatorname{ord}_{17}(5)=16$. Хватило **одного** возведения в степень.

### Идея алгоритма

Перебирать все делители $t$ не нужно — достаточно простых множителей $t$. Начинаем с $t=\varphi(n)$ и для каждого простого $q\mid\varphi(n)$ повторяем: пока $q\mid t$ и $a^{t/q}\equiv1\pmod n$, заменяем $t$ на $t/q$.

На каждом шаге $\operatorname{ord}(a)$ делит $t$: в начале это (3.11), а уменьшаем $t$ мы только тогда, когда $a^{t/q}\equiv1$, то есть по (3.10) когда $\operatorname{ord}(a)\mid t/q$. В конце ни один простой множитель уже нельзя убрать, значит, $t$ не имеет собственного делителя, кратного порядку, и $t=\operatorname{ord}(a)$.

### Сколько это стоит

Если разложение $\varphi(n)=q_1^{f_1}\cdots q_s^{f_s}$ **известно**, то алгоритм делает не более $f_1+\cdots+f_s\le\log_2\varphi(n)<L$ возведений в степень, каждое за полиномиальное время. Вся трудность — в разложении $\varphi(n)$. Именно так этот факт сформулирован в сводке [2, приложение A.2.2, п. 5, с. 815]: если разложение $p-1$ известно, порядок любого элемента вычисляется просто и эффективно.

Здесь важен один практический момент. Для простого модуля $p$ порядок группы $p-1$ известен без вычислений, а вызов `euler_phi(p)` раскладывал бы само $p$ пробным делением — для 61-битного $p$ это около $2^{30}$ шагов. Поэтому функция ниже принимает порядок группы `t` как необязательный аргумент.

<div style="background-color: #1e2021; padding: 20px; border-radius: 10px; border-left: 4px solid #007cba;">

### Задание 3.12. `multiplicative_order_fast`

Напишите функцию

```python
def multiplicative_order_fast(a, n, t=None):
    '''Порядок a по модулю n. t — известное кратное порядка (по умолчанию euler_phi(n)).'''
```

по алгоритму выше; простые множители $t$ получите через `decompose(t)`. Добавьте в `ZnZ` метод `multiplicative_order_fast(x)`, который её вызывает.

1. Проверьте `multiplicative_order_fast(5, 17) == 16`.
2. Проверьте совпадение с `multiplicative_order` для всех элементов $(\mathbb Z/n\mathbb Z)^*$ при $2\le n\le300$.
3. Вычислите порядок $3$ по модулю простого $M_{61}=2^{61}-1$, передав `t = 2**61 - 2`. Сколько времени это заняло? Сколько заняло `decompose(2**61 - 2)`? Оцените, не запуская, сколько заняло бы `euler_phi(2**61 - 1)` (подсказка: это разложение 61-битного простого пробным делением).

</div>

In [20]:
# Ваше решение:

# Быстрый порядок. Перебор из 3.10 делает до φ(n) умножений.
# Здесь t — известное кратное порядка (по Лагранжу им можно взять φ(n)).
# Достаточно простых множителей t. Пока q делит t и a^{t/q} ≡ 1,
# порядок делит t/q, и q из t вычёркивается. Когда вычёркивать больше нечего,
# t совпадает с порядком.

def multiplicative_order_fast(a, n, t=None):
    """Порядок a по модулю n. t — известное кратное порядка (по умолчанию euler_phi(n))."""
    if gcd(a, n) != 1:
        raise ValueError("a и n должны быть взаимно просты")
    # Без готового t порядок группы считается разложением n. Для простого p это
    # как раз дорогое пробное деление самого p, хотя p - 1 и так известен.
    if t is None:
        t = euler_phi(n)
    a %= n
    # Список простых снимаем сразу, до того как t начнёт уменьшаться.
    # Иначе разложение пришлось бы пересчитывать на каждом делении.
    for q in decompose(t):
        # Условие a^{t/q} ≡ 1 значит, что порядок делит t/q, свойство (3.10).
        # Пока степень q в t ещё есть, пробуем снять её ещё раз: у q^2, q^3, ...
        while t % q == 0 and mod_pow(a, t // q, n) == 1:
            t //= q
    return t


def multiplicative_order_fast_method(self, x):
    return multiplicative_order_fast(x, self.n)


ZnZ.multiplicative_order_fast = multiplicative_order_fast_method

assert multiplicative_order_fast(5, 17) == 16
print("ord_17(5) =", multiplicative_order_fast(5, 17))

# Сверка с медленным порядком на всех обратимых классах при n <= 300.
# Быстрый алгоритм и перебор обязаны вернуть одно k.
mismatches = []
for n in range(2, 301):
    group = ZnZ(n)
    for a in group.elems():
        if group.multiplicative_order(a) != multiplicative_order_fast(a, n):
            mismatches.append((n, a))
print(f"совпадение с перебором для всех элементов при n <= 300, расхождений: {len(mismatches)}")
assert not mismatches

# M_61 = 2^61 - 1 простое. Порядок группы равен p - 1 = 2^61 - 2,
# его и передаём в t. euler_phi(p) стал бы раскладывать само 61-битное простое.
M61 = 2**61 - 1
started = time.perf_counter()
order_3 = multiplicative_order_fast(3, M61, 2**61 - 2)
order_time = time.perf_counter() - started
started = time.perf_counter()
factors = decompose(2**61 - 2)
factor_time = time.perf_counter() - started
# Разложение 2^61 - 2 обязано собраться обратно, найденный порядок делит p - 1,
# и 3 в этой степени сравнимо с 1. Иначе алгоритм снял лишний простой множитель.
assert recompose(factors) == 2**61 - 2
assert mod_pow(3, order_3, M61) == 1
assert (2**61 - 2) % order_3 == 0
print(f"ord(3) по модулю M_61 = {order_3}")
print(f"время порядка: {order_time:.6f} с; decompose(2^61 - 2): {factor_time:.6f} с")
print("разложение 2^61 - 2:", factors)
print(
    "euler_phi(2^61 - 1) раскладывал бы само 61-битное простое пробным делением до √p ≈ 2^{30.5}.\n"
    "Это около 2^30 делений: уже не доля секунды, а десятки секунд.\n"
    "2^61 - 2 = 2(2^60 - 1) делится на маленькие простые, поэтому decompose заканчивается сразу.\n"
    "Порядок группы p - 1 для простого модуля известен без разложения p, его и передают в t."
)


ord_17(5) = 16
совпадение с перебором для всех элементов при n <= 300, расхождений: 0
ord(3) по модулю M_61 = 256204778801521550
время порядка: 0.000135 с; decompose(2^61 - 2): 0.000010 с
разложение 2^61 - 2: {2: 1, 3: 2, 5: 2, 7: 1, 11: 1, 13: 1, 31: 1, 41: 1, 61: 1, 151: 1, 331: 1, 1321: 1}
euler_phi(2^61 - 1) раскладывал бы само 61-битное простое пробным делением до √p ≈ 2^{30.5}.
Это около 2^30 делений: уже не доля секунды, а десятки секунд.
2^61 - 2 = 2(2^60 - 1) делится на маленькие простые, поэтому decompose заканчивается сразу.
Порядок группы p - 1 для простого модуля известен без разложения p, его и передают в t.


<div style="background-color: #212120; padding: 20px; border-radius: 10px; border-left: 4px solid #d39e00;">

### 🔬 Исследование 3.5. Перебор против быстрого алгоритма

**Гипотеза.** Как растёт время каждого метода, если битовая длина простого модуля увеличивается на 4?

**Ход.** Для каждого $L$ возьмите `p = random_prime(L)` и элемент $a$ с большим порядком (например, первый $a\ge2$, для которого `multiplicative_order_fast(a, p, p - 1) == p - 1`). Замерьте время `ZnZ(p).multiplicative_order(a)` и `multiplicative_order_fast(a, p, p - 1)`; медиана по трём простым.

*Замечание.* Если ваш `ZnZ.__init__` сразу строит список всех элементов, создание объекта для больших $p$ само станет перебором. Для этого исследования либо стройте список лениво (в момент вызова `elems()`), либо вызывайте перебор отдельной функцией.

| $L$, бит | перебор, с | быстрый алгоритм, с |
|---:|---:|---:|
| 12 | | |
| 16 | | |
| 20 | | |
| 24 | | |
| 61 ($M_{61}$) | — | |

**Вопросы.**

1. Во сколько раз растёт время перебора при $L\to L+4$? А быстрого алгоритма?
2. Что из нужного быстрому алгоритму становится трудным для 2048-битного $p$, если про $p-1$ ничего не известно? Как этого избегают при выборе параметров (подсказка — раздел 3.8)?

</div>

In [21]:
# Выполните исследование здесь:

from statistics import median

print(
    "Гипотеза. Перебор делает около p ≈ 2^L умножений, поэтому при L → L+4 время растёт примерно в 16 раз.\n"
    "Быстрый алгоритм делает O(L) возведений в степень и одно разложение p-1.\n"
    "На этих длинах p-1 короткий, так что его время почти не растёт."
)


# Для замера нужен элемент большого порядка, иначе перебор закончится слишком рано
# и не покажет цену худшего случая. Берём первый a >= 2, у которого порядок равен p - 1:
# это первообразный корень, и медленный алгоритм делает ровно p - 1 шаг.
def generator_mod(p):
    a = 2
    while multiplicative_order_fast(a, p, p - 1) != p - 1:
        a += 1
    return a


def median_pair(L, trials=3):
    brute_times = []
    fast_times = []
    for _ in range(trials):
        p = random_prime(L)
        a = generator_mod(p)
        # ZnZ(p) сам список элементов не строит. В замер входит только multiplicative_order.
        # Быстрому алгоритму передаём t = p - 1, чтобы он не раскладывал простое p.
        group = ZnZ(p)
        started = time.perf_counter()
        group.multiplicative_order(a)
        brute_times.append(time.perf_counter() - started)
        started = time.perf_counter()
        multiplicative_order_fast(a, p, p - 1)
        fast_times.append(time.perf_counter() - started)
    return median(brute_times), median(fast_times)


print(f"\n{'L':>4} {'перебор, с':>14} {'быстрый, с':>14}")
measured = []
for L in (12, 16, 20, 24):
    brute_time, fast_time = median_pair(L)
    measured.append((L, brute_time, fast_time))
    print(f"{L:4d} {brute_time:14.6f} {fast_time:14.6f}")

# Перебор порядка на M_61 — это около 2^61 умножений, его не запускаем.
# В таблице для 61 бита только быстрый столбец.
p = 2**61 - 1
started = time.perf_counter()
multiplicative_order_fast(3, p, p - 1)
print(f"{'61':>4} {'—':>14} {time.perf_counter() - started:14.6f}")

brute_ratios = [
    measured[i][1] / measured[i - 1][1] for i in range(1, len(measured))
]
fast_ratios = [
    measured[i][2] / measured[i - 1][2] for i in range(1, len(measured))
]
print(
    "\n1. Отношения перебора при L → L+4: "
    + ", ".join(f"{ratio:.1f}" for ratio in brute_ratios)
    + ".\n"
    "   Это близко к 16 = 2^4: порядок образующего равен p-1 и растёт как 2^L.\n"
    "   Отношения быстрого алгоритма: "
    + ", ".join(f"{ratio:.1f}" for ratio in fast_ratios)
    + ".\n"
    "   Там нет перебора порядка. Растёт только цена нескольких возведений в степень.\n"
    "\n2. Для 2048-битного p быстрому алгоритму нужно разложение p-1.\n"
    "   Если про p-1 ничего не известно, это разложение само трудное.\n"
    "   Его избегают выбором модуля: p строят так, чтобы разложение p-1 было известно заранее.\n"
    "   Безопасное простое из раздела 3.8 — крайний случай, p-1 = 2q с простым q."
)


Гипотеза. Перебор делает около p ≈ 2^L умножений, поэтому при L → L+4 время растёт примерно в 16 раз.
Быстрый алгоритм делает O(L) возведений в степень и одно разложение p-1.
На этих длинах p-1 короткий, так что его время почти не растёт.

   L     перебор, с     быстрый, с
  12       0.000095       0.000003
  16       0.001087       0.000002
  20       0.019606       0.000008
  24       0.310220       0.000013
  61              —       0.000154

1. Отношения перебора при L → L+4: 11.4, 18.0, 15.8.
   Это близко к 16 = 2^4: порядок образующего равен p-1 и растёт как 2^L.
   Отношения быстрого алгоритма: 0.9, 3.2, 1.6.
   Там нет перебора порядка. Растёт только цена нескольких возведений в степень.

2. Для 2048-битного p быстрому алгоритму нужно разложение p-1.
   Если про p-1 ничего не известно, это разложение само трудное.
   Его избегают выбором модуля: p строят так, чтобы разложение p-1 было известно заранее.
   Безопасное простое из раздела 3.8 — крайний случай, p-1 = 2q с простым 

## 3.6. Циклические группы и подгруппы

### Определение 3.5. Циклическая группа

Группа $G$ называется **циклической**, если в ней есть элемент $g$, степени которого дают все элементы группы:

$$
G=\langle g\rangle=\{\,g^k:\ k\in\mathbb Z\,\}.
\tag{3.16}
$$

Такой $g$ называется **образующим** (порождающим) элементом. Для конечной группы $G$ элемент $g$ образующий тогда и только тогда, когда $\operatorname{ord}(g)=|G|$.

**Пример.** $(\mathbb Z/9\mathbb Z)^*=\{1,2,4,5,7,8\}$ циклична: степени $2$ дают $2,4,8,7,5,1$. Элемент $5$ тоже образующий, а $4$ — нет: его степени $4,7,1$ образуют только часть группы.

### Подгруппы

В конечной группе любое непустое подмножество, замкнутое относительно умножения, является подгруппой. Степени одного элемента

$$
\langle a\rangle=\{1,a,a^2,\ldots,a^{\operatorname{ord}(a)-1}\}
\tag{3.17}
$$

образуют подгруппу ровно из $\operatorname{ord}(a)$ элементов.

### Что известно о циклических группах

Напомним факты из курса алгебры (свойства порядка — [3, теорема 1.6.31, с. 151–153], [4, теорема 9.4, с. 351]; число элементов порядка $d$ — [4, теорема 9.8, с. 357]; тождество (3.18) — [4, теорема 7.7, с. 244]). Пусть $G=\langle g\rangle$ — циклическая группа порядка $N$. Тогда:

- $\operatorname{ord}(g^k)=N/\gcd(k,N)$;
- для каждого делителя $d\mid N$ в $G$ есть **ровно одна** подгруппа порядка $d$, и она циклическая;
- элементов порядка $d$ ровно $\varphi(d)$; в частности, образующих элементов ровно $\varphi(N)$.

Подсчитав элементы группы по их порядкам, получаем тождество

$$
\sum_{d\mid N}\varphi(d)=N.
\tag{3.18}
$$

Для криптографии важно второе свойство: зная разложение $N$, мы точно знаем, какие подгруппы есть в группе. Именно по ним «рассыпается» дискретный логарифм в разделе 3.8.

<div style="background-color: #181a1b; padding: 20px; border-radius: 10px; border-left: 4px solid #007cba;">

### Задание 3.13. Подгруппа, порождённая элементом

Добавьте в `ZnZ` метод `subgroup(x)`, возвращающий список $[1,x,x^2,\ldots,x^{\operatorname{ord}(x)-1}]$.

```python
G = ZnZ(9)
print(G.subgroup(4))
print(G.subgroup(2))
```

Ожидаемый вывод:

```text
[1, 4, 7]
[1, 2, 4, 8, 7, 5]
```

</div>

In [22]:
# Ваше решение:

# Подгруппа, порождённая x: {1, x, x^2, ..., x^{ord(x)-1}}.
# Элементов ровно ord(x). Начинаем с x^0 = 1 и домножаем на x по одному разу.

def subgroup(self, x):
    x %= self.n
    # Длина списка известна заранее. Когда степень дойдёт до порядка, следующая была бы 1,
    # и список зациклился бы. Поэтому шагов ровно order, включая единицу.
    order = self.multiplicative_order(x)
    elements = []
    power = 1
    for _ in range(order):
        elements.append(power)
        power = (power * x) % self.n
    return elements


ZnZ.subgroup = subgroup

G = ZnZ(9)
print(G.subgroup(4))
print(G.subgroup(2))
assert G.subgroup(4) == [1, 4, 7]
assert G.subgroup(2) == [1, 2, 4, 8, 7, 5]


[1, 4, 7]
[1, 2, 4, 8, 7, 5]


<div style="background-color: #1f2021; padding: 20px; border-radius: 10px; border-left: 4px solid #007cba;">

### Задание 3.14. Все подгруппы $(\mathbb Z/27\mathbb Z)^*$

Группа $(\mathbb Z/27\mathbb Z)^*$ циклична и имеет порядок $18$.

1. Найдите все её подгруппы: соберите множества `frozenset(G.subgroup(x))` для всех $x$. Сколько их и каковы их порядки? Сравните с делителями числа $18$.
2. Для каждого делителя $d\mid18$ подсчитайте число элементов порядка $d$ и сравните с $\varphi(d)$.
3. Проверьте тождество (3.18) для всех $N\le1000$.

Для самопроверки: подгрупп ровно шесть, а их порядки — $1, 2, 3, 6, 9, 18$. Подгруппа порядка $9$:

```text
[1, 4, 16, 10, 13, 25, 19, 22, 7]
```

</div>

In [23]:
# Ваше решение:

G = ZnZ(27)
# (Z/27Z)* циклична порядка φ(27) = 18. Подгруппа, порождённая x, зависит только
# от множества степеней, не от того, каким x мы её породили.
# frozenset сравнивает множества, а не порядок записи. set из них убирает повторы:
# разные x часто порождают одну и ту же подгруппу.

subgroups = {frozenset(G.subgroup(x)) for x in G.elems()}
orders = sorted(len(subgroup) for subgroup in subgroups)
# В циклической группе ровно одна подгруппа на каждый делитель порядка.
# Порядки найденных подгрупп должны совпасть с делителями 18.
divisors_18 = [d for d in range(1, 19) if 18 % d == 0]
print("подгрупп:", len(subgroups))
print("их порядки:", orders)
print("делители 18:", divisors_18)
assert len(subgroups) == 6
assert orders == divisors_18
for subgroup in sorted(subgroups, key=len):
    print(len(subgroup), sorted(subgroup))
print("подгруппа порядка 9 как степени 4:", G.subgroup(4))
assert G.subgroup(4) == [1, 4, 16, 10, 13, 25, 19, 22, 7]

# Элементов порядка d в циклической группе ровно φ(d).
# Складывая φ(d) по d | 18, получаем 18: это тождество (3.18) на одном примере.
order_count = Counter(G.multiplicative_order(x) for x in G.elems())
print("\nэлементы порядка d и φ(d):")
for d in divisors_18:
    print(f"  d = {d}: найдено {order_count[d]}, φ(d) = {euler_phi(d)}")
    assert order_count[d] == euler_phi(d)


# Та же сумма для общего N <= 1000: перебор делителей и φ(d) по формуле.
def sum_phi_over_divisors(N):
    return sum(euler_phi(d) for d in range(1, N + 1) if N % d == 0)


bad = [N for N in range(1, 1001) if sum_phi_over_divisors(N) != N]
print("тождество Σ_{d|N} φ(d) = N для N <= 1000, расхождений:", len(bad))
assert not bad


подгрупп: 6
их порядки: [1, 2, 3, 6, 9, 18]
делители 18: [1, 2, 3, 6, 9, 18]
1 [1]
2 [1, 26]
3 [1, 10, 19]
6 [1, 8, 10, 17, 19, 26]
9 [1, 4, 7, 10, 13, 16, 19, 22, 25]
18 [1, 2, 4, 5, 7, 8, 10, 11, 13, 14, 16, 17, 19, 20, 22, 23, 25, 26]
подгруппа порядка 9 как степени 4: [1, 4, 16, 10, 13, 25, 19, 22, 7]

элементы порядка d и φ(d):
  d = 1: найдено 1, φ(d) = 1
  d = 2: найдено 1, φ(d) = 1
  d = 3: найдено 2, φ(d) = 2
  d = 6: найдено 2, φ(d) = 2
  d = 9: найдено 6, φ(d) = 6
  d = 18: найдено 6, φ(d) = 6
тождество Σ_{d|N} φ(d) = N для N <= 1000, расхождений: 0


<div style="background-color: #18191a; padding: 20px; border-radius: 10px; border-left: 4px solid #007cba;">

### Задание 3.15. Циклична ли группа

Добавьте в `ZnZ` метод `is_cyclic()`: группа циклична, если в ней есть элемент порядка `group_order()`.

```python
for n in (9, 19, 8, 15):
    G = ZnZ(n)
    print(n, G.group_order(), G.is_cyclic())
```

Ожидаемый вывод:

```text
9 6 True
19 18 True
8 4 False
15 8 False
```

Для $n=8$ найдите порядки всех элементов. Почему группа порядка $4$, в которой нет элемента порядка $4$, не может быть циклической?

</div>

In [24]:
# Ваше решение:

# Группа конечная и циклична ровно тогда, когда в ней есть элемент порядка |G|.
# |G| = φ(n) = group_order(). Достаточно найти хотя бы один такой элемент.

def is_cyclic(self):
    target = self.group_order()
    return any(self.multiplicative_order(a) == target for a in self.elems())


ZnZ.is_cyclic = is_cyclic

for n in (9, 19, 8, 15):
    G = ZnZ(n)
    print(n, G.group_order(), G.is_cyclic())

# (Z/8Z)* = {1, 3, 5, 7}, порядок группы 4. Циклическая группа порядка 4
# выглядела бы как {1, g, g^2, g^3} с ord(g) = 4. Если такого g нет, группа не циклическая.
orders_8 = {a: ZnZ(8).multiplicative_order(a) for a in ZnZ(8).elems()}
print("порядки в (Z/8Z)*:", orders_8)
print(
    "Циклическая группа порядка 4 имеет вид {1, g, g^2, g^3} с ord(g) = 4.\n"
    "В (Z/8Z)* такого g нет: порядки равны 1 или 2. Значит группа не циклическая."
)
assert orders_8 == {1: 1, 3: 2, 5: 2, 7: 2}


9 6 True
19 18 True
8 4 False
15 8 False
порядки в (Z/8Z)*: {1: 1, 3: 2, 5: 2, 7: 2}
Циклическая группа порядка 4 имеет вид {1, g, g^2, g^3} с ord(g) = 4.
В (Z/8Z)* такого g нет: порядки равны 1 или 2. Значит группа не циклическая.


## 3.7. Первообразные корни

### Определение 3.6. Первообразный корень

Число $g$, $\gcd(g,n)=1$, называется **первообразным корнем** по модулю $n$, если $\operatorname{ord}_n(g)=\varphi(n)$. Иначе говоря, первообразный корень — это образующий элемент группы $(\mathbb Z/n\mathbb Z)^*$, и он существует ровно тогда, когда эта группа циклична.

Для простого $p$ первообразный корень $g$ перечисляет все ненулевые классы:

$$
(\mathbb Z/p\mathbb Z)^*=\{1,g,g^2,\ldots,g^{p-2}\}.
\tag{3.19}
$$

Значит, каждое $h\not\equiv0\pmod p$ однозначно записывается как $h\equiv g^x$ с $0\le x<p-1$. Показатель $x$ называется **дискретным логарифмом** $h$ по основанию $g$ (раздел 3.8).

### Когда первообразные корни существуют

Кольцо $\mathbb Z/m\mathbb Z$ является полем тогда и только тогда, когда $m=p$ простое: только в этом случае все ненулевые классы обратимы (критерий (1.23) ноутбука 1).

<div style="border-top: 1px solid #999; border-bottom: 1px solid #999; padding: 12px 4px; margin: 18px 0;">

<strong>Теорема 3.2. Существование первообразных корней</strong>

1. (Гаусс) Для любого простого $p$ группа $(\mathbb Z/p\mathbb Z)^*$ циклична, и первообразных корней по модулю $p$ ровно $\varphi(p-1)$.
2. Первообразный корень по модулю $n>1$ существует тогда и только тогда, когда

$$
n\in\{\,2,\ 4,\ p^t,\ 2p^t\,\},
$$

где $p$ — нечётное простое, $t\ge1$.

</div>

Формулировки — [1, теорема 1.31 и замечание 1.33, с. 33–34], [3, теоремы 1.6.34–1.6.35, с. 152–155], [4, следствие 9.8.1, с. 357; теорема 9.15, с. 367], [5, теорема 11.10, с. 247; §11.6, с. 263–266]. Доказательства — в курсе теории чисел; ключевой довод в п. 1 состоит в том, что многочлен $x^d-1$ имеет в поле не более $d$ корней. Мы проверим обе части теоремы экспериментально.

### Как проверить, что $g$ — первообразный корень

Применим идею раздела 3.5: $g$ — первообразный корень, если его порядок не является собственным делителем $\varphi(n)$. Любой собственный делитель $\varphi(n)$ делит $\varphi(n)/q$ хотя бы для одного простого $q\mid\varphi(n)$, поэтому по (3.10):

<div style="border-top: 1px solid #999; border-bottom: 1px solid #999; padding: 12px 4px; margin: 18px 0;">

<strong>Теорема 3.3. Критерий первообразного корня</strong>

Пусть $\gcd(g,n)=1$. Число $g$ является первообразным корнем по модулю $n$ тогда и только тогда, когда

$$
g^{\varphi(n)/q}\not\equiv1\pmod n\qquad\text{для каждого простого } q\mid\varphi(n).
\tag{3.20}
$$

</div>

Проверка (3.20) требует столько возведений в степень, сколько различных простых делителей у $\varphi(n)$, — и снова разложения $\varphi(n)$. Для простого $p$ это разложение $p-1$. Критерий (3.20) для простого модуля — [5, предложение 11.11, с. 248].

<div style="background-color: #232526; padding: 20px; border-radius: 10px; border-left: 4px solid #007cba;">

### Задание 3.16. Проверка и поиск первообразного корня

1. Напишите `is_primitive_root(g, n, t=None)` по критерию (3.20); `t` — порядок группы, по умолчанию `euler_phi(n)`.
2. Напишите `primitive_root(n)`, возвращающую **наименьший** первообразный корень или `None`, если его нет.
3. Добавьте в `ZnZ` метод `primitive_roots()`, возвращающий список всех первообразных корней (пустой список, если их нет).

Проверьте:

```python
print(ZnZ(9).primitive_roots())
print(ZnZ(8).primitive_roots())
print(ZnZ(19).primitive_roots(), euler_phi(18))
print(primitive_root(106), ZnZ(106).primitive_roots()[:3])
```

Ожидаемый вывод:

```text
[2, 5]
[]
[2, 3, 10, 13, 14, 15] 6
3 [3, 5, 19]
```

Сравните список для $n=19$ с ответом задания 3.11 (п. 4). Примеры с $n=106$ и $n=34$ взяты из [3, упражнение 1.6.4 и пример 1.6.37, с. 152–153].

4. Покажите, что $g=5$ — первообразный корень по модулю $34$ и что степени $5^1,\ldots,5^{16}\bmod 34$ дают в точности $(\mathbb Z/34\mathbb Z)^*$.

</div>

In [25]:
# Ваше решение:

# Теорема 3.2, пункт 2: первообразный корень есть ровно при n = 2, 4, p^k или 2p^k,
# где p — нечётное простое. Эта функция только узнаёт форму модуля.
# Сам поиск корня ниже на неё не опирается: иначе сверка с теоремой была бы круговой.

def has_primitive_root_modulus(n):
    """Теорема 3.2: корень есть ровно при n = 2, 4, p^k или 2p^k."""
    if n in (2, 4):
        return True
    # Снимаем не больше одной двойки. Если после этого число всё ещё чётное,
    # n делится на 4 и не равно 4: это не 2p^k.
    odd_part = n
    if odd_part % 2 == 0:
        odd_part //= 2
        if odd_part % 2 == 0:
            return False
    if odd_part == 1:
        return False
    # Оставшаяся нечётная часть должна быть степенью одного простого.
    # Делим на него, пока делится. Если что-то осталось, простых множителей два или больше.
    prime = smallest_factor(odd_part)
    while odd_part % prime == 0:
        odd_part //= prime
    return odd_part == 1


# Критерий (3.20). g — первообразный корень, когда его порядок равен φ(n).
# Любой меньший делитель φ(n) делит φ(n)/q хотя бы для одного простого q | φ(n).
# Поэтому достаточно проверить, что g^{φ(n)/q} ≢ 1 для каждого такого q.
# t — готовый порядок группы. Для простого p это p - 1, и p заново не раскладывается.
def is_primitive_root(g, n, t=None):
    if gcd(g, n) != 1:
        return False
    if t is None:
        t = euler_phi(n)
    g %= n
    for q in decompose(t):
        # Уже для одного q степень равна 1, значит порядок делит t/q и меньше t.
        if mod_pow(g, t // q, n) == 1:
            return False
    return True


# Наименьший корень: кандидаты идут от 1 вверх, первый успех критерия и есть минимум.
# Если корня нет, критерий отклонит каждый обратимый класс и функция вернёт None.
# Теорему 3.2 здесь не вызываем, чтобы задание 3.17 могло сверить поиск с теоремой.
def primitive_root(n):
    """Наименьший первообразный корень по модулю n или None.

    Существование здесь не подменяется теоремой 3.2: если корня нет,
    критерий (3.20) отклоняет каждый обратимый класс.
    """
    if n < 2:
        return None
    group_order = euler_phi(n)
    for g in range(1, n):
        if gcd(g, n) == 1 and is_primitive_root(g, n, group_order):
            return g
    return None


# Все первообразные корни группы, по возрастанию. Пустой список — группа не циклична.
# group_order() передаём в критерий, чтобы φ(n) не считалось заново на каждом элементе.
def primitive_roots(self):
    group_order = self.group_order()
    return [g for g in self.elems() if is_primitive_root(g, self.n, group_order)]


ZnZ.primitive_roots = primitive_roots

print(ZnZ(9).primitive_roots())
print(ZnZ(8).primitive_roots())
print(ZnZ(19).primitive_roots(), euler_phi(18))
print(primitive_root(106), ZnZ(106).primitive_roots()[:3])
assert ZnZ(9).primitive_roots() == [2, 5]
assert ZnZ(8).primitive_roots() == []
assert ZnZ(19).primitive_roots() == [2, 3, 10, 13, 14, 15]
assert euler_phi(18) == 6
assert primitive_root(106) == 3
assert ZnZ(106).primitive_roots()[:3] == [3, 5, 19]

# 5^1, ..., 5^16 по модулю 34. φ(34) = φ(2·17) = 16, поэтому 16 степеней —
# это как раз вся подгруппа, порождённая 5. Если 5 — корень, множество совпадёт с группой.
# 5^16 ≡ 1, единица в списке есть. Сортировка нужна только для сравнения множеств.
powers = []
power = 1
for _ in range(16):
    power = (power * 5) % 34
    powers.append(power)
print("степени 5^1 .. 5^16 mod 34:", powers)
print("это в точности (Z/34Z)*:", sorted(powers) == ZnZ(34).elems())
print("5 — первообразный корень по модулю 34:", is_primitive_root(5, 34))
assert sorted(powers) == ZnZ(34).elems()
assert is_primitive_root(5, 34)


[2, 5]
[]
[2, 3, 10, 13, 14, 15] 6
3 [3, 5, 19]
степени 5^1 .. 5^16 mod 34: [5, 25, 23, 13, 31, 19, 27, 33, 29, 9, 11, 21, 3, 15, 7, 1]
это в точности (Z/34Z)*: True
5 — первообразный корень по модулю 34: True


<div style="background-color: #151616; padding: 20px; border-radius: 10px; border-left: 4px solid #007cba;">

### Задание 3.17. Для каких $n$ есть первообразные корни

1. Найдите все $2\le n\le30$, для которых существует первообразный корень, и сравните список с теоремой 3.2 (п. 2).
2. Для тех же $n$ вычислите **показатель группы** $\lambda(n)$ — наибольший порядок элемента $(\mathbb Z/n\mathbb Z)^*$ (он же наименьшее общее кратное порядков всех элементов, `math.lcm`). Составьте таблицу $n,\ \varphi(n),\ \lambda(n)$.
3. Убедитесь, что $\lambda(n)\mid\varphi(n)$ и что $\lambda(n)=\varphi(n)$ ровно для тех $n$, у которых есть первообразный корень.

Для самопроверки: $\lambda(8)=2$, $\lambda(15)=4$, $\lambda(24)=2$, а $\varphi$ этих чисел равны $4, 8, 8$.

</div>

In [26]:
# Ваше решение:

import math

# Два независимых списка. with_root — где поиск критерием (3.20) реально нашёл корень.
# by_theorem — где модуль имеет вид 2, 4, p^k или 2p^k. Они должны совпасть.

with_root = [n for n in range(2, 31) if primitive_root(n) is not None]
by_theorem = [n for n in range(2, 31) if has_primitive_root_modulus(n)]
print("есть первообразный корень:", with_root)
print("по теореме 3.2:         ", by_theorem)
assert with_root == by_theorem

print(f"\n{'n':>4} {'φ(n)':>6} {'λ(n)':>6} {'λ=φ':>6}")
for n in range(2, 31):
    group = ZnZ(n)
    # λ(n), показатель группы, — наибольший порядок элемента.
    # Он же наименьшее общее кратное всех порядков: math.lcm это и считает.
    # λ(n) делит φ(n) по Лагранжу. Равенство значит, что есть элемент порядка φ(n),
    # то есть группа циклична и первообразный корень есть.
    orders = [group.multiplicative_order(a) for a in group.elems()]
    carmichael = math.lcm(*orders)
    phi = group.group_order()
    print(f"{n:4d} {phi:6d} {carmichael:6d} {str(carmichael == phi):>6}")
    assert phi % carmichael == 0
    assert (carmichael == phi) == (primitive_root(n) is not None)

for n, expected in ((8, 2), (15, 4), (24, 2)):
    group = ZnZ(n)
    carmichael = math.lcm(*(group.multiplicative_order(a) for a in group.elems()))
    assert carmichael == expected
print("Самопроверка: λ(8)=2, λ(15)=4, λ(24)=2, а φ этих модулей равны 4, 8 и 8.")
print("λ(n) делит φ(n) всегда. Равенство λ(n)=φ(n) выполнено ровно тогда, когда группа циклична.")


есть первообразный корень: [2, 3, 4, 5, 6, 7, 9, 10, 11, 13, 14, 17, 18, 19, 22, 23, 25, 26, 27, 29]
по теореме 3.2:          [2, 3, 4, 5, 6, 7, 9, 10, 11, 13, 14, 17, 18, 19, 22, 23, 25, 26, 27, 29]

   n   φ(n)   λ(n)    λ=φ
   2      1      1   True
   3      2      2   True
   4      2      2   True
   5      4      4   True
   6      2      2   True
   7      6      6   True
   8      4      2  False
   9      6      6   True
  10      4      4   True
  11     10     10   True
  12      4      2  False
  13     12     12   True
  14      6      6   True
  15      8      4  False
  16      8      4  False
  17     16     16   True
  18      6      6   True
  19     18     18   True
  20      8      4  False
  21     12      6  False
  22     10     10   True
  23     22     22   True
  24      8      2  False
  25     20     20   True
  26     12     12   True
  27     18     18   True
  28     12      6  False
  29     28     28   True
  30      8      4  False
Самопроверка: λ(8)=

<div style="border-left: 3px solid #666; padding-left: 18px; margin: 18px 0;">

**Функция Кармайкла.** Величина $\lambda(n)$ из задания 3.17 называется функцией Кармайкла. По определению $a^{\lambda(n)}\equiv1$ для **всех** обратимых $a$, и $\lambda(n)$ — наименьший такой показатель. Для $n=pq$

$$
\lambda(pq)=\operatorname{lcm}(p-1,\,q-1),
$$

и в стандартах RSA секретный показатель часто вычисляют по модулю $\lambda(n)$, а не $\varphi(n)$ (ср. [3, с. 364–365]). Мы вернёмся к этому в ноутбуке о RSA. Определение и свойства $\lambda(n)$ — [3, определение 1.4.7, с. 80–81; теорема 1.6.17, с. 125–127], [4, §9.6, теорема 9.22, с. 385–387] (там $\lambda(n)$ называется минимальным универсальным показателем).

</div>

<div style="background-color: #22211f; padding: 20px; border-radius: 10px; border-left: 4px solid #d39e00;">

### 🔬 Исследование 3.6. Как часто встречаются первообразные корни

**Гипотеза.** По теореме 3.2 доля первообразных корней среди элементов $(\mathbb Z/p\mathbb Z)^*$ равна $\varphi(p-1)/(p-1)$. Предскажите: насколько большим бывает наименьший первообразный корень для $p<10^5$? Для какой доли простых $p$ число $2$ — первообразный корень?

**Ход.** Для всех простых $3\le p<10^5$ (их список даёт `primes` из задания 3.0) найдите наименьший первообразный корень $g(p)$ с помощью `primitive_root(p)`, а также долю $\varphi(p-1)/(p-1)$.

**Результаты** (не зависят от компьютера, заполните по своим вычислениям):

| величина | значение |
|---|---:|
| число простых $3\le p<10^5$ | |
| сколько $p$ имеют $g(p)=2$; $g(p)=3$; $g(p)=5$ | |
| наибольшее $g(p)$ и при каком $p$ | |
| доля $p$, для которых $2$ — первообразный корень | |
| минимум и максимум $\varphi(p-1)/(p-1)$ | |
| среднее $\varphi(p-1)/(p-1)$ | |

**Вопросы.**

1. Почему $g(p)$ никогда не бывает равным $4$ (и вообще точному квадрату)?
2. При каких $p$ доля $\varphi(p-1)/(p-1)$ максимальна и чему равна? Как устроено $p-1$ у этих простых?
3. Сколько случайных $g$ в среднем придётся проверить, чтобы найти первообразный корень, если доля равна $\varphi(p-1)/(p-1)$? Почему поэтому поиск первообразного корня практичен даже для 2048-битного $p$ — при условии, что известно разложение $p-1$?
4. Сравните долю простых, для которых $2$ — первообразный корень, с числом $0{,}3739\ldots$ (константа Артина). Гипотеза Артина утверждает, что именно к этой доле стремится частота; она доказана лишь в предположении расширенной гипотезы Римана ([3, гипотеза 1.6.1, с. 155]; см. также [4, с. 357]). Чем ваш эксперимент отличается от доказательства?

Метода предсказать наименьший первообразный корень простого $p$ не известно ([3, с. 152–153]); таблица наименьших первообразных корней для $n\le1017$ приведена в [3, таблица 1.21, с. 154–155]. Похожее упражнение для $p<100$ — [1, упражнение 1.32, с. 53].

</div>

In [27]:
# Выполните исследование здесь:

from fractions import Fraction

print(
    "Гипотеза. Доля первообразных корней равна φ(p-1)/(p-1) и обычно составляет десятые доли,\n"
    "поэтому наименьший корень g(p) маленький: чаще всего 2, 3 или 5, и вряд ли больше нескольких десятков.\n"
    "Долю простых, для которых корень равен 2, жду около константы Артина 0.37."
)

# Простые 3 <= p < 10^5. primes(n) возвращает p <= n, поэтому верхняя граница 99999.
# Двойку убираем: для p = 2 доля φ(1)/1 особая, а исследование про нечётные простые.
ps = [p for p in primes(10**5 - 1) if p >= 3]
roots = []
shares = []
for p in ps:
    roots.append(primitive_root(p))
    # Доля первообразных корней среди 1..p-1 равна φ(p-1)/(p-1).
    # Fraction, а не float: максимум 1/2 должен сравниваться точно, без округления.
    shares.append(Fraction(euler_phi(p - 1), p - 1))

root_count = Counter(roots)
max_root = max(roots)
p_of_max_root = ps[roots.index(max_root)]
max_share = max(shares)
min_share = min(shares)
p_of_max_share = [p for p, share in zip(ps, shares) if share == max_share]
p_of_min_share = ps[shares.index(min_share)]
mean_share = sum(shares) / len(shares)
# Если доля корней равна s, случайный кандидат подходит с вероятностью s.
# Среднее число попыток до первого успеха — 1/s = (p-1)/φ(p-1).
mean_trials = sum(1 / share for share in shares) / len(shares)

print("\nвеличина | значение")
print(f"число простых 3 <= p < 10^5 | {len(ps)}")
print(f"g(p)=2; g(p)=3; g(p)=5 | {root_count[2]}; {root_count[3]}; {root_count[5]}")
print(f"наибольшее g(p) и p | {max_root} при p = {p_of_max_root}")
print(f"доля p с g(p)=2 | {root_count[2] / len(ps):.4f}")
print(f"минимум и максимум φ(p-1)/(p-1) | {float(min_share):.6f} и {float(max_share):.6f}")
print(f"среднее φ(p-1)/(p-1) | {float(mean_share):.4f}")
# Ни один наименьший корень не квадрат. Квадрат g = a^2 даёт
# g^{(p-1)/2} = a^{p-1} ≡ 1, и критерий для q = 2 уже не выполнен.
assert all(isqrt(root) ** 2 != root for root in roots)

print(
    "\n1. Точный квадрат не бывает первообразным корнем. Если g = a^2, то\n"
    "   g^{(p-1)/2} = a^{p-1} ≡ 1 (mod p), и критерий (3.20) для q = 2 не выполнен.\n"
    "   Поэтому g(p) не равно 4 и вообще ни одному квадрату.\n"
    f"\n2. Максимум φ(p-1)/(p-1) равен {max_share} при p = {p_of_max_share}.\n"
    "   У каждого из них p-1 — степень двойки: любой нечётный простой q | p-1\n"
    "   умножил бы долю на (1 - 1/q) < 1. Минимум на этом отрезке достигается при\n"
    f"   p = {p_of_min_share}, p-1 = {p_of_min_share - 1} = {decompose(p_of_min_share - 1)}:\n"
    "   в p-1 сидят самые маленькие простые, и произведение (1-1/q) получается наименьшим.\n"
    f"\n3. Случайных кандидатов в среднем {float(mean_trials):.2f}: это среднее (p-1)/φ(p-1).\n"
    "   Каждый кандидат — несколько возведений в степень, полиномиальных по L.\n"
    "   Поиск остаётся практичным и для 2048-битного p, если разложение p-1 уже известно\n"
    "   и критерий (3.20) можно проверить.\n"
    f"\n4. Доля g(p)=2 на p < 10^5 равна {root_count[2] / len(ps):.4f}, константа Артина ≈ 0.3739.\n"
    "   Это конечный подсчёт. Гипотеза Артина говорит о пределе при p → ∞\n"
    "   и доказана только в предположении расширенной гипотезы Римана.\n"
    "   Совпасть на отрезке до 10^5 — наблюдение, а не доказательство предела."
)


Гипотеза. Доля первообразных корней равна φ(p-1)/(p-1) и обычно составляет десятые доли,
поэтому наименьший корень g(p) маленький: чаще всего 2, 3 или 5, и вряд ли больше нескольких десятков.
Долю простых, для которых корень равен 2, жду около константы Артина 0.37.

величина | значение
число простых 3 <= p < 10^5 | 9591
g(p)=2; g(p)=3; g(p)=5 | 3603; 2212; 1366
наибольшее g(p) и p | 44 при p = 71761
доля p с g(p)=2 | 0.3757
минимум и максимум φ(p-1)/(p-1) | 0.195569 и 0.500000
среднее φ(p-1)/(p-1) | 0.3743

1. Точный квадрат не бывает первообразным корнем. Если g = a^2, то
   g^{(p-1)/2} = a^{p-1} ≡ 1 (mod p), и критерий (3.20) для q = 2 не выполнен.
   Поэтому g(p) не равно 4 и вообще ни одному квадрату.

2. Максимум φ(p-1)/(p-1) равен 1/2 при p = [3, 5, 17, 257, 65537].
   У каждого из них p-1 — степень двойки: любой нечётный простой q | p-1
   умножил бы долю на (1 - 1/q) < 1. Минимум на этом отрезке достигается при
   p = 78541, p-1 = 78540 = {2: 2, 3: 1, 5: 1, 7: 1, 11: 1, 17: 1}

## 3.8. Криптографическое применение: дискретный логарифм

### Определение 3.7. Дискретный логарифм

Пусть $p$ — простое, $g$ — первообразный корень по модулю $p$, $h\not\equiv0\pmod p$. **Дискретным логарифмом** $h$ по основанию $g$ называется число $x$, $0\le x<p-1$, для которого

$$
g^x\equiv h\pmod p.
\tag{3.21}
$$

Вычислить $h=g^x$ по $x$ легко: `mod_pow` делает $O(L)$ умножений. Обратная задача — найти $x$ по $h$ — в общем случае считается трудной. На этой асимметрии построены протоколы Диффи—Хеллмана и Эль-Гамаля: $x$ — секретный ключ, $h=g^x$ — открытый. См. [1, §2.2, с. 62–64], [2, §10.5, с. 403–406], [4, §9.4, с. 368], [5, §11.5, с. 257–258]. Таблица степеней $2^x\bmod19$ из [5, с. 257] даёт ответ задания 3.18.

<div style="background-color: #232628; padding: 20px; border-radius: 10px; border-left: 4px solid #007cba;">

### Задание 3.18. Дискретный логарифм перебором

Напишите `discrete_log_bruteforce(g, h, p)`, которая перебирает $x=0,1,2,\ldots$ и возвращает первое $x$ с $g^x\equiv h$ (или `None`). Используйте последовательное умножение, а не `mod_pow` на каждом шаге.

```python
print(discrete_log_bruteforce(2, 13, 19))
print(mod_pow(2, discrete_log_bruteforce(2, 13, 19), 19))
```

Ожидаемый вывод: `5` и `13`.

</div>

In [28]:
# Ваше решение:

# Дискретный логарифм: наименьшее x >= 0 с g^x ≡ h (mod p).
# Степени считаем домножением на g, а не mod_pow на каждом шаге:
# иначе каждый шаг сам стоил бы O(log x) умножений и замер перебора был бы нечестным.
# x = 0 соответствует g^0 = 1. Если за p шагов совпадения нет, h не степень g.

def discrete_log_bruteforce(g, h, p):
    """Наименьшее x >= 0 с g^x ≡ h (mod p). Последовательное умножение, без mod_pow в цикле."""
    h %= p
    # 1 % p на случай p = 1 не нужен, но оставляет power в диапазоне остатков сразу.
    # Для обычного простого p это просто 1.
    power = 1 % p
    x = 0
    while x < p:
        if power == h:
            return x
        power = (power * g) % p
        x += 1
    return None


found = discrete_log_bruteforce(2, 13, 19)
print(found)
print(mod_pow(2, found, 19))
assert found == 5
assert mod_pow(2, found, 19) == 13


5
13


<div style="background-color: #31302d; padding: 20px; border-radius: 10px; border-left: 4px solid #d39e00;">

### 🔬 Исследование 3.7. Прямая и обратная задачи

**Гипотеза.** Во сколько раз вырастет время перебора дискретного логарифма при $L\to L+4$? А время `mod_pow`?

**Ход.** Для каждого $L$ возьмите `p = random_prime(L)`, `g = primitive_root(p)`, случайный секрет $x$, $h=g^x\bmod p$. Замерьте время `mod_pow(g, x, p)` и `discrete_log_bruteforce(g, h, p)`; медиана по трём запускам.

**Часть A. Замеры.**

| $L$, бит | `mod_pow`, с | перебор, с |
|---:|---:|---:|
| 12 | | |
| 16 | | |
| 20 | | |
| 24 | | |

**Часть B. Экстраполяция.** Оцените время одного шага перебора и умножьте на среднее число шагов $\approx p/2\approx2^{L-1}$ для $L=64,\ 128,\ 2048$. Сравните с временем `mod_pow` для 2048-битного модуля (его можно замерить на простом $M_{2203}$ из исследования 3.4).

**Вопросы.**

1. Как растут два времени по $L$? Назовите это различие одним словом каждое.
2. Перебор — не лучший метод: существуют алгоритмы порядка $\sqrt p$ шагов, а для $(\mathbb Z/p\mathbb Z)^*$ — и существенно более быстрые. Как это меняет вашу оценку из части B и почему на практике используют $p$ длиной 2048 бит и более? Обзор оценок — [1, §2.6–2.7, с. 75–81], [3, §2.4.1, с. 255], [5, §11.5.1, с. 258–260].

</div>

In [29]:
# Выполните исследование здесь:

from statistics import median
import math

print(
    "Гипотеза. Перебор в среднем делает около p/2 ≈ 2^{L-1} шагов и при L → L+4 растёт примерно в 16 раз.\n"
    "mod_pow делает O(L) умножений по модулю, поэтому его время растёт медленно, полиномиально по L."
)


# Один опыт: простое p из L бит, первообразный корень g, случайный секрет x,
# открытое h = g^x. Прямая задача — mod_pow, обратная — перебор.
# mod_pow на этих длинах короче шума одного вызова, поэтому усредняем 300 одинаковых.
# Перебор один: он и есть измеряемая величина, повторять его внутри опыта не нужно.
def one_trial(L):
    p = random_prime(L)
    g = primitive_root(p)
    secret = random.randrange(0, p - 1)
    public = mod_pow(g, secret, p)
    repeats = 300
    started = time.perf_counter()
    for _ in range(repeats):
        mod_pow(g, secret, p)
    pow_time = (time.perf_counter() - started) / repeats
    started = time.perf_counter()
    found = discrete_log_bruteforce(g, public, p)
    brute_time = time.perf_counter() - started
    assert found == secret
    # Число домножений в переборе равно x (на ответе цикл останавливается до умножения).
    # Если x = 0, делить время на ноль нельзя: цена шага тогда не из чего считать.
    return pow_time, brute_time, max(secret, 1)


print(f"\n{'L':>4} {'mod_pow, с':>14} {'перебор, с':>14}")
measured = []
step_samples = []
for L in (12, 16, 20, 24):
    pow_samples = []
    brute_samples = []
    for _ in range(3):
        pow_time, brute_time, steps = one_trial(L)
        pow_samples.append(pow_time)
        brute_samples.append(brute_time)
        step_samples.append(brute_time / steps)
    row = (L, median(pow_samples), median(brute_samples))
    measured.append(row)
    print(f"{L:4d} {row[1]:14.6e} {row[2]:14.6f}")

# Цена одного шага — медиана по всем опытам, не по одной длине:
# умножение чисел до 24 бит стоит примерно одинаково.
# Дальше среднее число шагов для длины L берём как p/2 ≈ 2^{L-1}.
step = median(step_samples)
print(f"\nодин шаг перебора ≈ {step:.3e} с")
print(f"{'L':>6} {'перебор, с':>16} {'перебор, годы':>16}")
for L in (64, 128, 2048):
    # 2^{L-1} не помещается в float, поэтому считаем десятичный логарифм.
    # 2^{2047} в float не помещается. log10(t) = log10(шаг) + (L-1)·log10(2)
    # переводит оценку сразу в степень десятки.
    log10_seconds = math.log10(step) + (L - 1) * math.log10(2)
    log10_years = log10_seconds - math.log10(365.25 * 24 * 3600)
    print(
        f"{L:6d} {'10^' + format(log10_seconds, '.1f'):>16} "
        f"{'10^' + format(log10_years, '.1f'):>16}"
    )

# Прямую задачу на криптографической длине мерить на случайном 2048-битном простом
# нельзя: проверять его простоту мы ещё не умеем. M_2203 простое по условию исследования 3.4.
# Показатель p-2 той же длины, что и настоящий секретный показатель.
mersenne = 2**2203 - 1
base = 3
started = time.perf_counter()
mod_pow(base, mersenne - 2, mersenne)
mersenne_time = time.perf_counter() - started
print(f"mod_pow на M_2203, один вызов: {mersenne_time:.4f} с")

brute_ratios = [measured[i][2] / measured[i - 1][2] for i in range(1, len(measured))]
print(
    "\n1. Отношения перебора при L → L+4: "
    + ", ".join(f"{ratio:.1f}" for ratio in brute_ratios)
    + ".\n"
    "   Секрет x случаен, поэтому медиана трёх запусков скачет вокруг множителя 16.\n"
    "   Рост при этом экспоненциальный: за три шага по 4 бита время уходит из микросекунд в десятые доли.\n"
    "   mod_pow на этих длинах остаётся в микросекундах, а на 2203-битном модуле занимает доли секунды.\n"
    "   Это полиномиальный рост.\n"
    "\n2. Алгоритм порядка √p заменяет 2^{L-1} на 2^{L/2}. Для L = 64 это уже около 2^32 шагов,\n"
    "   для L = 128 — 2^64, для L = 2048 оценка части B остаётся недостижимой.\n"
    "   Для (Z/pZ)* известны и более быстрые методы, но они тоже не полиномиальны по L.\n"
    "   Поэтому на практике берут p длиной 2048 бит и больше: даже ускоренный перебор\n"
    "   не становится посильным."
)


Гипотеза. Перебор в среднем делает около p/2 ≈ 2^{L-1} шагов и при L → L+4 растёт примерно в 16 раз.
mod_pow делает O(L) умножений по модулю, поэтому его время растёт медленно, полиномиально по L.

   L     mod_pow, с     перебор, с
  12   5.266900e-07       0.000094
  16   7.296900e-07       0.000847
  20   9.925233e-07       0.007778
  24   1.304587e-06       0.127266

один шаг перебора ≈ 2.984e-08 с
     L       перебор, с    перебор, годы
    64          10^11.4           10^3.9
   128          10^30.7          10^23.2
  2048         10^608.7         10^601.2
mod_pow на M_2203, один вызов: 0.0340 с

1. Отношения перебора при L → L+4: 9.0, 9.2, 16.4.
   Секрет x случаен, поэтому медиана трёх запусков скачет вокруг множителя 16.
   Рост при этом экспоненциальный: за три шага по 4 бита время уходит из микросекунд в десятые доли.
   mod_pow на этих длинах остаётся в микросекундах, а на 2203-битном модуле занимает доли секунды.
   Это полиномиальный рост.

2. Алгоритм порядка √p заменяе

### Почему важна структура $p-1$

Группа $(\mathbb Z/p\mathbb Z)^*$ имеет порядок $p-1$, и по разделу 3.6 в ней есть подгруппа для каждого делителя $p-1$. Если $q\mid p-1$, то возведение в степень $(p-1)/q$ переводит всю группу в подгруппу порядка $q$:

$$
h^{(p-1)/q}\equiv\bigl(g^{(p-1)/q}\bigr)^{x}\equiv\bigl(g^{(p-1)/q}\bigr)^{x\bmod q}\pmod p.
\tag{3.22}
$$

Элемент $g^{(p-1)/q}$ имеет порядок $q$, поэтому (3.22) позволяет найти **остаток $x\bmod q$** перебором всего $q$ вариантов. Если все простые делители $p-1$ малы, то, найдя остатки $x$ по каждому из них, можно восстановить $x$ целиком по КТО из ноутбука 1. Этот подход называют методом Полига—Хеллмана ([1, §2.9, теорема 2.32, с. 86–91], [3, §2.4.2, с. 258]).

<div style="background-color: #2a2c2d; padding: 20px; border-radius: 10px; border-left: 4px solid #007cba;">

### Задание 3.19. Утечка секрета через малые подгруппы

1. **Один бит утекает всегда.** Возьмите $q=2$ в (3.22). Покажите, что $h^{(p-1)/2}\equiv1$ при чётном $x$ и $\equiv-1$ при нечётном. Проверьте на $p=19$, $g=2$ для всех $x$.
2. **Весь секрет.** Ниже дано простое $p=200\,560\,490\,131$ (38 бит), его первообразный корень $g=34$ и открытый ключ $h=g^x\bmod p$. Число $p-1=2\cdot3\cdot5\cdot7\cdot11\cdot13\cdot17\cdot19\cdot23\cdot29\cdot31$ раскладывается только на малые простые.
   - Для каждого $q\mid p-1$ найдите $x\bmod q$ по (3.22) перебором $q$ вариантов.
   - Восстановите $x$ по остаткам с помощью `crt` из `crypto_math`.
   - Проверьте, что `mod_pow(g, x, p) == h`.
3. Сколько всего возведений в степень вам понадобилось? Сколько шагов сделал бы `discrete_log_bruteforce` (оцените по исследованию 3.7)?

</div>

In [30]:
p = 200560490131
g = 34
h = 6960430086

assert mod_pow(g, (p - 1) // 2, p) == p - 1   # хотя бы g — не квадрат; полную проверку даст is_primitive_root
print(decompose(p - 1))

# Ваше решение:
print("1. g — первообразный корень, поэтому g^{(p-1)/2} ≡ -1 (mod p).")
print("   h = g^x даёт h^{(p-1)/2} ≡ (-1)^x: это 1 при чётном x и -1 при нечётном.")
# 1. Для первообразного корня g^{(p-1)/2} ≡ -1. Поэтому h^{(p-1)/2} ≡ (-1)^x.
# Чётный x даёт 1, нечётный даёт -1, то есть p-1 в остатках. Проверяем все x для p = 19, g = 2.
# Один бит секрета виден всегда: чётность x. Это частный случай q = 2 в формуле (3.22).
for secret in range(18):
    public = mod_pow(2, secret, 19)
    legendre = mod_pow(public, (19 - 1) // 2, 19)
    # -1 mod 19 равно 18. Сравниваем остатки, а не отрицательное число.
    expected = 1 if secret % 2 == 0 else 18
    assert legendre == expected
print("На p = 19, g = 2 равенство проверено для всех x = 0..17.")

# Полная проверка, что g = 34 порождает всю группу. t = p - 1 известен,
# раскладывать 38-битное простое p для этого не нужно.
assert is_primitive_root(g, p, p - 1)
residues = []
moduli = []
exponentiations = 0
multiplications = 0
# 2. Для каждого простого q | p-1 элемент g^{(p-1)/q} имеет порядок q.
# h^{(p-1)/q} ≡ (g^{(p-1)/q})^{x mod q}. Перебор k = 0..q-1 находит x mod q.
# Показатели q в разложении равны 1, поэтому старших степеней q^e нет
# и одного остатка по модулю q достаточно.
for q in decompose(p - 1):
    # Два настоящих возведения в степень на каждый простой. Дальше перебор остатка
    # идёт домножением, это уже не степень.
    base = mod_pow(g, (p - 1) // q, p)
    target = mod_pow(h, (p - 1) // q, p)
    exponentiations += 2
    power = 1
    found = None
    for k in range(q):
        if power == target:
            found = k
            break
        power = (power * base) % p
        multiplications += 1
    residues.append(found)
    moduli.append(q)
    print(f"x mod {q} = {found}")

# Модули q попарно просты, их произведение равно p - 1.
# Китайская теорема склеивает остатки в единственный x по модулю p - 1.
# crt возвращает наименьший неотрицательный, он и лежит в 0..p-2.
x = crt(residues, moduli)
print("x =", x)
print("mod_pow(g, x, p) == h:", mod_pow(g, x, p) == h)
assert mod_pow(g, x, p) == h

# Оценка перебора из задания 3.18 на этом же p: сколько стоит одно домножение.
# До найденного x шагов примерно x, до половины группы — около p/2.
sample = 200_000
acc = 1
started = time.perf_counter()
for _ in range(sample):
    acc = (acc * g) % p
step = (time.perf_counter() - started) / sample
print(f"возведений в степень: {exponentiations}, домножений в переборе остатков: {multiplications}")
print(
    f"один шаг перебора ≈ {step:.3e} с. До найденного x это около {step * x:.3e} с,\n"
    f"до p/2 — около {step * (p / 2):.3e} с. Полиг—Хеллман обошёлся {exponentiations} степенями,\n"
    "потому что все простые делители p-1 маленькие и остатки склеиваются китайской теоремой."
)


{2: 1, 3: 1, 5: 1, 7: 1, 11: 1, 13: 1, 17: 1, 19: 1, 23: 1, 29: 1, 31: 1}
1. g — первообразный корень, поэтому g^{(p-1)/2} ≡ -1 (mod p).
   h = g^x даёт h^{(p-1)/2} ≡ (-1)^x: это 1 при чётном x и -1 при нечётном.
На p = 19, g = 2 равенство проверено для всех x = 0..17.
x mod 2 = 1
x mod 3 = 0
x mod 5 = 1
x mod 7 = 3
x mod 11 = 4
x mod 13 = 1
x mod 17 = 2
x mod 19 = 12
x mod 23 = 15
x mod 29 = 27
x mod 31 = 7
x = 108022074681
mod_pow(g, x, p) == h: True
возведений в степень: 22, домножений в переборе остатков: 73
один шаг перебора ≈ 7.348e-08 с. До найденного x это около 7.938e+03 с,
до p/2 — около 7.369e+03 с. Полиг—Хеллман обошёлся 22 степенями,
потому что все простые делители p-1 маленькие и остатки склеиваются китайской теоремой.


### Безопасные простые и подгруппа простого порядка

Задание 3.19 показывает, что для дискретного логарифма важна не длина $p$, а **наибольший простой делитель** $p-1$. Отсюда стандартный выбор параметров.

Простое $p$ называется **безопасным** (*safe prime*), если $q=(p-1)/2$ тоже простое. Тогда порядок группы $p-1=2q$ и подгруппы в ней только порядков $1,2,q,2q$. Работу ведут в подгруппе порядка $q$: её образующий получается как $g=a^2\bmod p$ для любого $a\not\equiv0,\pm1$. Тогда по малой теореме Ферма (3.13) $g^q\equiv a^{p-1}\equiv1$, поэтому по (3.10) порядок $g$ делит простое $q$; а так как $g\ne1$, он равен $q$. В подгруппе простого порядка нет меньших подгрупп, и приём из задания 3.19 ничего не даёт.

В [2, §10.4.1, с. 402] протокол Диффи—Хеллмана описан именно в подгруппе простого порядка $q\mid p-1$ группы $(\mathbb Z/p\mathbb Z)^*$ (для ориентира: $p$ около 2048 бит, $q$ около 256 бит). Безопасные простые упоминаются в [2, упражнение 10.19, с. 421], а критерий первообразного корня для них — в [1, упражнение 1.33, с. 54].

<div style="background-color: #282b2d; padding: 20px; border-radius: 10px; border-left: 4px solid #007cba;">

### Задание 3.20. Безопасное простое и мини-протокол Диффи—Хеллмана

1. Найдите наименьшее безопасное простое $p\ge10^6$: перебирайте простые $p$ функцией `next_prime` и проверяйте простоту $(p-1)/2$ функцией `is_prime_fast`. Для самопроверки: $p=1\,000\,667$.
2. Возьмите случайное $a$, $2\le a\le p-2$, и $g=a^2\bmod p$. Проверьте, что $\operatorname{ord}(g)=q$ (используйте `multiplicative_order_fast(g, p, p - 1)`).
3. **Взгляд вперёд: Диффи—Хеллман.** Алиса выбирает секрет $x_A$, Боб — секрет $x_B$ (случайные числа от $1$ до $q-1$). Они обмениваются $A=g^{x_A}$ и $B=g^{x_B}$ по открытому каналу. Вычислите $K_A=B^{x_A}$ и $K_B=A^{x_B}$ и убедитесь, что $K_A=K_B$. Объясните равенство одной строкой.
4. Какую задачу должен решить противник, видящий $p,g,A,B$, чтобы найти $K$? Почему в этой подгруппе не работает приём из задания 3.19?

Протокол Диффи—Хеллмана, его безопасность и выбор параметров подробно разбираются в отдельном ноутбуке; элементарное изложение — [5, §12.1, с. 273–275], [4, §8.6, с. 338].

</div>

In [31]:
# Ваше решение:

# Безопасное простое: и p, и q = (p-1)/2 простые. Тогда порядок группы p-1 = 2q,
# и подгруппы только порядков 1, 2, q, 2q.
# next_prime(10^6 - 1) — первое простое, которое не меньше 10^6.
# Дальше идём по простым, пока (p-1)/2 тоже не окажется простым.

p = next_prime(10**6 - 1)
while not is_prime_fast((p - 1) // 2):
    p = next_prime(p)
q = (p - 1) // 2
print("наименьшее безопасное простое p >= 10^6:", p)
print("q = (p - 1) / 2 =", q, "простое:", is_prime_fast(q))
assert p == 1_000_667

# a от 2 до p-2, поэтому a ≢ 0 и a ≢ ±1. Тогда g = a^2 не равен 0 и не равен 1.
# По Ферма g^q = a^{p-1} ≡ 1, значит порядок g делит простое q.
# Раз g ≠ 1, порядок не 1 и равен q. Это образующий подгруппы простого порядка.
a = random.randrange(2, p - 1)
g = mod_pow(a, 2, p)
order = multiplicative_order_fast(g, p, p - 1)
print(f"a = {a}, g = a^2 mod p = {g}, ord(g) = {order}")
assert g != 1 and order == q

# Секреты Диффи—Хеллмана берутся из 1..q-1: показатель 0 дал бы открытый ключ 1.
# Алиса публикует A = g^{x_A}, Боб публикует B = g^{x_B}.
# Общий ключ K = g^{x_A x_B}. Алиса считает его как B^{x_A}, Боб как A^{x_B}.
x_a = random.randrange(1, q)
x_b = random.randrange(1, q)
A = mod_pow(g, x_a, p)
B = mod_pow(g, x_b, p)
K_a = mod_pow(B, x_a, p)
K_b = mod_pow(A, x_b, p)
print(f"x_A = {x_a}, x_B = {x_b}")
print(f"K_A = {K_a}")
print(f"K_B = {K_b}")
print("K_A = K_B:", K_a == K_b)
assert K_a == K_b
print(
    "K_A = B^{x_A} = (g^{x_B})^{x_A} = g^{x_A x_B} = (g^{x_A})^{x_B} = A^{x_B} = K_B.\n"
    "Противник видит p, g, A, B и хочет получить K = g^{x_A x_B}.\n"
    "Это задача Диффи—Хеллмана в подгруппе порядка q; её решают через дискретный логарифм A или B.\n"
    "Приём задания 3.19 здесь не уменьшает секрет: порядок подгруппы — само простое q,\n"
    "меньших подгрупп нет, и остаток показателя по модулю q — это уже весь показатель."
)


наименьшее безопасное простое p >= 10^6: 1000667
q = (p - 1) / 2 = 500333 простое: True
a = 882752, g = a^2 mod p = 679927, ord(g) = 500333
x_A = 317159, x_B = 335792
K_A = 930477
K_B = 930477
K_A = K_B: True
K_A = B^{x_A} = (g^{x_B})^{x_A} = g^{x_A x_B} = (g^{x_A})^{x_B} = A^{x_B} = K_B.
Противник видит p, g, A, B и хочет получить K = g^{x_A x_B}.
Это задача Диффи—Хеллмана в подгруппе порядка q; её решают через дискретный логарифм A или B.
Приём задания 3.19 здесь не уменьшает секрет: порядок подгруппы — само простое q,
меньших подгрупп нет, и остаток показателя по модулю q — это уже весь показатель.


## 3.9. Перенос в `crypto_math`

Часть функций ноутбука понадобится в ноутбуках о Диффи—Хеллмане, Эль-Гамале и RSA. Перенесём их в библиотеку курса; класс `ZnZ` остаётся учебным инструментом этого ноутбука.

<div style="background-color: #21262a; padding: 20px; border-radius: 10px; border-left: 4px solid #007cba;">

### Задание 3.21. Пополнение библиотеки

1. Перенесите в `crypto_math/number_theory.py` функции

   `smallest_factor`, `decompose`, `euler_phi`, `multiplicative_order_fast` (под именем `multiplicative_order`), `is_primitive_root`, `primitive_root`.

   Функция `euler_phi` в библиотеке может вычисляться прямо через `decompose`, без `mult_function`.
2. Добавьте в `tests/test_number_theory.py` по одной проверке свойств: `recompose(decompose(n)) == n`; $a^{\varphi(n)}\equiv1$ для случайных обратимых $a$; `multiplicative_order(a, n)` делит `euler_phi(n)`; $\sum_{d\mid n}\varphi(d)=n$.
3. Перезапустите kernel и убедитесь, что функции импортируются из `crypto_math`.

</div>

In [32]:
# Ваше решение:

# В crypto_math/number_theory.py перенесены
# smallest_factor, decompose, euler_phi, multiplicative_order,
# is_primitive_root, primitive_root.
# euler_phi считается прямо через decompose, без mult_function.
# Класс ZnZ в библиотеку не входил: он остаётся инструментом этого ноутбука.
# В tests/test_number_theory.py добавлены проверки свойств из заданий 3.0 и 3.21.

# Импорт с другими именами, чтобы не затереть decompose и primitive_root этого ноутбука:
# ниже мы сравниваем библиотечные функции с теми, что определены в ячейках.
import subprocess
import sys
from pathlib import Path

from crypto_math.number_theory import (
    decompose as library_decompose,
    euler_phi as library_euler_phi,
    is_primitive_root as library_is_primitive_root,
    multiplicative_order,
    primitive_root as library_primitive_root,
)

# Короткая сверка контрактов перед pytest. 100 = 2^2 * 5^2, φ(9) = 6, φ(1) = 1.
# Порядок 5 по модулю 17 равен 16, значит 5 — первообразный корень, и наименьший
# корень по модулю 19 равен 2. На 2..60 библиотека и ноутбук обязаны выбрать один и тот же корень.
assert library_decompose(100) == {2: 2, 5: 2}
assert library_euler_phi(9) == 6 and library_euler_phi(1) == 1
assert multiplicative_order(5, 17) == 16
assert library_is_primitive_root(3, 17)
assert library_primitive_root(19) == 2
assert all(library_primitive_root(n) == primitive_root(n) for n in range(2, 61))
print("функции ноутбука импортируются из crypto_math")

# Ядро ноутбука может быть запущено из каталога 3/, а не из корня проекта.
# Поднимаемся к родителям, пока не найдём pyproject.toml рядом с пакетом crypto_math.
root = Path.cwd().resolve()
for candidate in (root, *root.parents):
    if (candidate / "pyproject.toml").exists() and (candidate / "crypto_math").is_dir():
        root = candidate
        break
else:
    raise RuntimeError("корень проекта MOZI не найден")

completed = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    cwd=root,
    check=True,
    capture_output=True,
    text=True,
)
print(completed.stdout.strip())
print("pytest прошёл")


функции ноутбука импортируются из crypto_math
........................................................................ [ 17%]
........................................................................ [ 34%]
........................................................................ [ 51%]
........................................................................ [ 68%]
........................................................................ [ 85%]
..............................................................           [100%]
422 passed in 0.62s
pytest прошёл


### Сверка с SageMath

Ячейку ниже выполните в ноутбуке с ядром **SageMath 10.x** после переноса функций. В обычном ядре Python ячейка лишь сообщит, что Sage недоступен.

In [33]:
import random

try:
    from sage.all import (
        factor as sage_factor,
        euler_phi as sage_euler_phi,
        Mod as sage_Mod,
        primitive_root as sage_primitive_root,
        is_prime as sage_is_prime,
    )
except ImportError:
    sage_available = False
else:
    sage_available = True

if not sage_available:
    print("SageMath недоступен в этом ядре. Переключите ядро на SageMath 10.x и выполните ячейку снова.")
else:
    from crypto_math.number_theory import (
        decompose, euler_phi, multiplicative_order, primitive_root, gcd,
    )

    rng = random.Random(2026)
    for _ in range(int(300)):
        n = rng.randrange(2, int(10) ** int(9))
        assert decompose(n) == {int(p): int(e) for p, e in sage_factor(n)}
        assert euler_phi(n) == int(sage_euler_phi(n))
        a = rng.randrange(1, n)
        if gcd(a, n) == 1 and n < int(10) ** int(6):
            assert multiplicative_order(a, n) == int(sage_Mod(a, n).multiplicative_order())

    for p in range(int(3), int(20000)):
        if sage_is_prime(p):
            g = primitive_root(p)
            assert int(sage_Mod(g, p).multiplicative_order()) == p - 1
            # Sage возвращает некоторый первообразный корень; наш — наименьший
            assert g <= int(sage_primitive_root(p))

    print("Сверка с SageMath пройдена.")

SageMath недоступен в этом ядре. Переключите ядро на SageMath 10.x и выполните ячейку снова.


<div style="background-color: #201e22; padding: 20px; border-radius: 10px; border-left: 4px solid #7653a6;">

### 🧩 Итоги ноутбука 3

**Что вычисляется быстро** (полиномиально по $L$):

- возведение в степень и обратный элемент — двумя способами (раздел 3.4);
- порядок элемента и проверка первообразного корня — **если известно разложение** порядка группы (разделы 3.5, 3.7).

**Что требует перебора** (экспоненциально по $L$ для изученных методов):

- разложение $n=pq$ и, как следствие, вычисление $\varphi(n)$ (разделы 3.1, 3.3);
- дискретный логарифм в $(\mathbb Z/p\mathbb Z)^*$ (раздел 3.8).

**Что из этого следует для криптографии:**

$$
\text{RSA}\leftarrow\text{трудность разложения } n=pq,\qquad
\text{DH, Эль-Гамаль}\leftarrow\text{трудность дискретного логарифма}.
$$

Параметры выбирают так, чтобы у «быстрых» алгоритмов было всё нужное (известно разложение $p-1$), а у противника — нет (подгруппа большого простого порядка).

**Пополнение `crypto_math/number_theory.py`** (в начале ноутбука — `is_prime_fast`, `primes`, `next_prime` из ноутбука 2; в конце — функции этого ноутбука):

```text
smallest_factor()   decompose()   euler_phi()
multiplicative_order()   is_primitive_root()   primitive_root()
```

**Дальше — ноутбук 4:** тест Ферма как следствие теоремы 3.1, числа Кармайкла, тест Миллера—Рабина, числа Мерсенна и генерация больших простых чисел.

</div>

## 3.10. Литература

### Основная литература

Ссылки в тексте ноутбука даны в виде [номер, раздел, страницы].

1. Hoffstein J., Pipher J., Silverman J. H. *An Introduction to Mathematical Cryptography*. — New York : Springer, 2008 (1st ed.). Страницы указаны по первому изданию.
   - §1.3, с. 22 — функция Эйлера и группа $(\mathbb Z/m\mathbb Z)^*$; теорема 1.21, с. 27 — основная теорема арифметики;
   - теорема 1.25, с. 30 — малая теорема Ферма; предложение 1.30, с. 32 — порядок делит $p-1$; §1.5, теорема 1.31, с. 33–34 — первообразные корни и их число $\varphi(p-1)$;
   - упражнения 1.31–1.33, с. 53–54 — элемент порядка $q$, первообразные корни, безопасные простые;
   - §2.2–2.3, с. 62–68 — задача дискретного логарифма и протокол Диффи—Хеллмана; предложение 2.14, с. 74 — теорема Лагранжа;
   - §2.6–2.7, с. 75–81 — трудность дискретного логарифма, перебор (предложение 2.20) и «шаг младенца — шаг великана»;
   - §2.9, с. 86–91 — алгоритм Полига—Хеллмана (теорема 2.32);
   - теорема 3.1, с. 114 — формула Эйлера для $pq$; замечание 3.10, с. 121–122 — знание $(p-1)(q-1)$ равносильно разложению $N=pq$.
2. Boneh D., Shoup V. *A Graduate Course in Applied Cryptography*. — Version 0.4, 2017.
   - §10.4, с. 401–402 — протокол Диффи—Хеллмана в подгруппе простого порядка $q\mid p-1$;
   - §10.5, с. 403–408 — предположение о трудности дискретного логарифма; пример подгруппы простого порядка в $\mathbb Z_p^*$;
   - упражнение 10.19, с. 421 — безопасные простые ($q=(p-1)/2$ простое);
   - приложение A.2.2, с. 815 — цикличность $\mathbb Z_p^*$, порядок, теорема Лагранжа, вычисление порядка при известном разложении $p-1$.
3. Yan S. Y. *Number Theory for Computing*. — 2nd ed. — Berlin : Springer, 2002.
   - теорема 1.2.8, с. 28 — основная теорема арифметики;
   - §1.4.1–1.4.2, с. 63–69 — мультипликативные функции, $\tau(n)$, $\sigma(n)$; теорема 1.4.10, с. 72–73 — чётные совершенные числа;
   - §1.4.4, с. 78–81 — функции Эйлера и Кармайкла;
   - теоремы 1.6.15–1.6.17, с. 125–127 — теоремы Ферма, Эйлера и Кармайкла;
   - §1.6.7, с. 150–155 — порядок и первообразные корни: свойства порядка (теорема 1.6.31), примеры для $n=34, 46$, упражнение о втором первообразном корне по модулю $106$, таблица 1.21 наименьших первообразных корней для $n\le1017$, критерий существования (теорема 1.6.35), гипотеза Артина;
   - §2.3.2, с. 232 — пробное деление; §2.4.1–2.4.2, с. 255–258 — алгоритмы «шаг младенца — шаг великана» и Сильвера—Полига—Хеллмана.

4. Rosen K. H. *Elementary Number Theory and Its Applications*. — 6th ed. — Boston : Addison-Wesley (Pearson), 2011.
   - §6.3, теорема 6.14, с. 234–236 — теорема Эйлера; §7.1–7.2, с. 239–253 — функция Эйлера (теоремы 7.4–7.5, тождество 7.7), $\tau$ и $\sigma$; §7.3, с. 256–260 — совершенные числа, тест Люка—Лемера;
   - §8.6, с. 338 — протокол Диффи—Хеллмана;
   - §9.1–9.3, с. 347–367 — порядок (теоремы 9.1, 9.4), число элементов порядка $d$ (теорема 9.8), гипотеза Артина (с. 357), критерий существования первообразного корня (теорема 9.15);
   - §9.4, с. 368–377 — дискретные логарифмы (индексы); §9.6, с. 385–389 — минимальный универсальный показатель $\lambda(n)$.
5. Kraft J. S., Washington L. C. *An Introduction to Number Theory with Cryptography*. — 2nd ed. — Boca Raton : CRC Press, 2018.
   - §8.1–8.2, с. 189–197 — теоремы Ферма и Эйлера, функция $\varphi$;
   - §11.1–11.2, с. 239–249 — порядок и первообразные корни, критерий (предложение 11.11); §11.5, с. 257–262 — дискретный логарифм, «шаг младенца — шаг великана»; §11.6, с. 263–266 — существование первообразных корней;
   - §12.1, с. 273–275 — протокол Диффи—Хеллмана; §14.1, с. 319 — пробное деление.
6. Herkommer M. A. *Number Theory: A Programmer's Guide*. — New York : McGraw-Hill, 1999.
   - §3.0–3.4, с. 49–69 — разложение пробным делением, $\varphi$ подсчётом НОД и через разложение, $\tau$, $\sigma$ (программы на C);
   - §4.4, с. 96–101 — порядок и первообразные корни (программа таблицы порядков).
7. Weissman M. H. *An Illustrated Theory of Numbers*. — Providence, RI : AMS, 2017; сопровождающий ноутбук P3wNT Notebook 4 — источник разделов 3.1–3.2.

### Дополнительная литература

8. Shoup V. *A Computational Introduction to Number Theory and Algebra*. — 2nd ed. — Cambridge University Press, 2009.
9. Мартынов Л. М. *Алгебра и теория чисел для криптографии* : учебное пособие. — СПб. : Лань, 2020.
10. Рябко Б. Я., Фионов А. Н. *Криптографические методы защиты информации* : учебное пособие для вузов. — М. : Горячая линия—Телеком, 2005.
11. [SageMath Reference Manual: Elementary Number Theory](https://doc.sagemath.org/html/en/reference/rings_standard/).